# Probing Exoplanetary Rings with Asterodensity Profiling: A PhotoRing Analysis of Kepler-51

Paper: https://arxiv.org/abs/2609.25234

# Part 1. Super-Puff Exoplanet — or a Ringed Planet?

In [ ]:
# ============================================================
# SUPER-PUFF OR RINGED PLANET?
# Inspired by Zuluaga et al. (2026)
# "Probing Exoplanetary Rings with Asterodensity Profiling:
#  A PhotoRing Analysis of Kepler-51"
#
# One self-contained Jupyter cell
# Requires: numpy, matplotlib, ffmpeg
#
# STORY
#   0–4 s      True planet + inclined rings
#   4–8 s      Transit across the star
#   8–13.2 s   Ringless interpretation of the same transit
#   13.2–14 s  Transition
#   14–20 s    Final comparison
#
# IMPORTANT
# Geometry and light curve are SCHEMATIC.
# This is not a fitted reconstruction of Kepler-51b.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle, Ellipse
from IPython.display import Video, display


# ============================================================
# OUTPUT
# ============================================================

OUTPUT = "mp4"       # "webm" or "mp4"

FPS = 30
DURATION = 20
NFRAMES = FPS * DURATION
DPI = 120

if OUTPUT == "webm":
    OUTFILE = "25234_super_puff_or_ringed_planet.webm"
else:
    OUTFILE = "25234_super_puff_or_ringed_planet.mp4"


# ============================================================
# HELPERS
# ============================================================

def clamp(x, a=0.0, b=1.0):
    return np.minimum(np.maximum(x, a), b)


def smoothstep(x):
    x = clamp(x)
    return x * x * (3.0 - 2.0 * x)


BG = "#02050b"


# ============================================================
# FIGURE
# ============================================================

fig = plt.figure(
    figsize=(12.8, 7.2),
    facecolor=BG
)


# ============================================================
# GLOBAL TITLE
# ============================================================

main_title = fig.text(
    0.5, 0.955,
    "SUPER-PUFF OR RINGED PLANET?",
    ha="center",
    va="center",
    color="white",
    fontsize=21,
    fontweight="bold",
    zorder=200
)

main_subtitle = fig.text(
    0.5, 0.916,
    "Unresolved rings can inflate the radius inferred from a transit",
    ha="center",
    va="center",
    color="#B7D6E8",
    fontsize=12,
    zorder=200
)


# ============================================================
# LEFT PANEL — TRANSIT GEOMETRY
# ============================================================

ax = fig.add_axes(
    [0.035, 0.15, 0.55, 0.70],
    zorder=10
)

ax.set_facecolor(BG)

ax.set_xlim(
    -1.65,
    1.65
)

ax.set_ylim(
    -1.15,
    1.15
)

ax.set_aspect("equal")
ax.axis("off")


# ============================================================
# STAR
# ============================================================

star_glow_1 = Circle(
    (0, 0),
    0.98,
    facecolor="#F5C868",
    edgecolor="none",
    alpha=0.055,
    zorder=1
)

star_glow_2 = Circle(
    (0, 0),
    0.88,
    facecolor="#FFD67A",
    edgecolor="none",
    alpha=0.10,
    zorder=2
)

star = Circle(
    (0, 0),
    0.76,
    facecolor="#F2C05E",
    edgecolor="#FFE7A3",
    linewidth=1.3,
    alpha=0.96,
    zorder=3
)

star_inner = Circle(
    (-0.10, 0.08),
    0.61,
    facecolor="#FFD87B",
    edgecolor="none",
    alpha=0.42,
    zorder=4
)

ax.add_patch(star_glow_1)
ax.add_patch(star_glow_2)
ax.add_patch(star)
ax.add_patch(star_inner)


# ============================================================
# TRUE PLANET + RINGS
# ============================================================

planet_x = -1.28
planet_y = 0.05

TRUE_R = 0.135

# Ring projected ellipse.
RING_W = 0.62
RING_H = 0.235
RING_ANGLE = -18


ring_outer = Ellipse(
    (planet_x, planet_y),
    RING_W,
    RING_H,
    angle=RING_ANGLE,
    facecolor="#8BCDE5",
    edgecolor="#D5F4FF",
    linewidth=1.3,
    alpha=0.28,
    zorder=11
)

ring_gap = Ellipse(
    (planet_x, planet_y),
    0.39,
    0.145,
    angle=RING_ANGLE,
    facecolor=BG,
    edgecolor="#83B8CB",
    linewidth=0.8,
    alpha=1.0,
    zorder=12
)

planet = Circle(
    (planet_x, planet_y),
    TRUE_R,
    facecolor="#376A87",
    edgecolor="#D5F5FF",
    linewidth=1.2,
    alpha=1.0,
    zorder=13
)

planet_highlight = Circle(
    (planet_x - 0.035, planet_y + 0.035),
    TRUE_R * 0.55,
    facecolor="#69A9C9",
    edgecolor="none",
    alpha=0.34,
    zorder=14
)

ax.add_patch(ring_outer)
ax.add_patch(ring_gap)
ax.add_patch(planet)
ax.add_patch(planet_highlight)


# ============================================================
# INFERRED RINGLESS PLANET
# ============================================================

APPARENT_R = 0.295

inferred_planet = Circle(
    (planet_x, planet_y),
    APPARENT_R,
    facecolor="#477F9B",
    edgecolor="#DDF8FF",
    linewidth=1.6,
    linestyle="--",
    alpha=0,
    zorder=20
)

ax.add_patch(inferred_planet)


# ============================================================
# LABELS
# ============================================================

true_label = ax.text(
    planet_x,
    -0.50,
    "TRUE PLANET + RINGS",
    ha="center",
    va="center",
    color="#DDF5FF",
    fontsize=11,
    fontweight="bold",
    zorder=30
)

true_sub = ax.text(
    planet_x,
    -0.64,
    "smaller planetary disk",
    ha="center",
    va="center",
    color="#8EB7C9",
    fontsize=9,
    zorder=30
)

inferred_label = ax.text(
    0,
    -0.92,
    "RINGLESS MODEL",
    ha="center",
    va="center",
    color="#AEEBFF",
    fontsize=12,
    fontweight="bold",
    alpha=0,
    zorder=30
)


# ============================================================
# RIGHT PANEL — TRANSIT LIGHT CURVE
# ============================================================

curve_ax = fig.add_axes(
    [0.625, 0.47, 0.32, 0.29],
    zorder=10
)

curve_ax.set_facecolor(
    "#06101A"
)

phase = np.linspace(
    -1.25,
    1.25,
    700
)


# ------------------------------------------------------------
# SCHEMATIC TRANSIT CURVE
# ------------------------------------------------------------

depth = 0.035

edge = 0.72
soft = 0.085

left_edge = (
    1 /
    (
        1 +
        np.exp(
            -(phase + edge) / soft
        )
    )
)

right_edge = (
    1 /
    (
        1 +
        np.exp(
            (phase - edge) / soft
        )
    )
)

transit_shape = (
    left_edge *
    right_edge
)

flux = (
    1.0 -
    depth *
    transit_shape
)


curve_line, = curve_ax.plot(
    phase,
    flux,
    color="#E2F8FF",
    lw=2.3
)

curve_ax.axhline(
    1.0,
    color="#5B7787",
    lw=0.8,
    alpha=0.5
)

curve_ax.set_xlim(
    -1.25,
    1.25
)

curve_ax.set_ylim(
    0.955,
    1.006
)

curve_ax.set_xlabel(
    "TRANSIT PHASE",
    color="#AFC9D8",
    fontsize=10,
    fontweight="bold"
)

curve_ax.set_ylabel(
    "NORMALIZED FLUX",
    color="#AFC9D8",
    fontsize=9,
    fontweight="bold"
)

curve_ax.tick_params(
    colors="#9AB6C6",
    labelsize=8
)

for spine in curve_ax.spines.values():
    spine.set_color("#52738A")

curve_ax.grid(
    color="#52738A",
    alpha=0.15
)

curve_ax.set_title(
    "OBSERVED TRANSIT",
    color="#DDF5FF",
    fontsize=12,
    fontweight="bold"
)


# ------------------------------------------------------------
# TRANSIT DEPTH
# ------------------------------------------------------------

depth_line, = curve_ax.plot(
    [0.88, 0.88],
    [1.0 - depth, 1.0],
    color="#7DE1FF",
    lw=2.2,
    alpha=0
)

depth_cap1, = curve_ax.plot(
    [0.82, 0.94],
    [1.0, 1.0],
    color="#7DE1FF",
    lw=1.5,
    alpha=0
)

depth_cap2, = curve_ax.plot(
    [0.82, 0.94],
    [1.0 - depth, 1.0 - depth],
    color="#7DE1FF",
    lw=1.5,
    alpha=0
)

depth_text = curve_ax.text(
    0.72,
    0.982,
    "TRANSIT\nDEPTH",
    color="#9CEAFF",
    fontsize=9,
    fontweight="bold",
    ha="center",
    va="center",
    alpha=0
)


# ============================================================
# RIGHT PANEL — INTERPRETATION
# ============================================================

info_ax = fig.add_axes(
    [0.625, 0.15, 0.32, 0.22],
    zorder=10
)

info_ax.set_facecolor(BG)
info_ax.axis("off")


info1 = info_ax.text(
    0.5, 0.82,
    "OCCULTING AREA",
    ha="center",
    va="center",
    color="#DCEFFA",
    fontsize=12,
    fontweight="bold",
    alpha=0,
    transform=info_ax.transAxes
)

arrow1 = info_ax.text(
    0.5, 0.61,
    "↓",
    ha="center",
    va="center",
    color="#72DFFF",
    fontsize=18,
    fontweight="bold",
    alpha=0,
    transform=info_ax.transAxes
)

info2 = info_ax.text(
    0.5, 0.43,
    "INFERRED RADIUS",
    ha="center",
    va="center",
    color="#DCEFFA",
    fontsize=12,
    fontweight="bold",
    alpha=0,
    transform=info_ax.transAxes
)

arrow2 = info_ax.text(
    0.5, 0.24,
    "↓",
    ha="center",
    va="center",
    color="#72DFFF",
    fontsize=18,
    fontweight="bold",
    alpha=0,
    transform=info_ax.transAxes
)

info3 = info_ax.text(
    0.5, 0.07,
    "INFERRED BULK DENSITY",
    ha="center",
    va="center",
    color="#A9EEFF",
    fontsize=12,
    fontweight="bold",
    alpha=0,
    transform=info_ax.transAxes
)


# ============================================================
# PHASE TEXT
# ============================================================

phase_text = fig.text(
    0.305, 0.855,
    "",
    ha="center",
    va="center",
    color="white",
    fontsize=16,
    fontweight="bold",
    zorder=50
)

phase_sub = fig.text(
    0.305, 0.818,
    "",
    ha="center",
    va="center",
    color="#B7D3E7",
    fontsize=11,
    zorder=50
)


# ============================================================
# FINAL SCENE — DARK BACKGROUND
# ============================================================

result_bg = fig.add_axes(
    [0.025, 0.075, 0.95, 0.80],
    zorder=70
)

result_bg.set_facecolor(
    "#000207"
)

result_bg.set_xticks([])
result_bg.set_yticks([])

for spine in result_bg.spines.values():
    spine.set_visible(False)

result_bg.set_visible(False)
result_bg.set_alpha(0)


# ============================================================
# FINAL SCENE — TRUE OBJECT
# ============================================================

true_ax = fig.add_axes(
    [0.095, 0.32, 0.33, 0.43],
    zorder=80
)

true_ax.set_facecolor(
    "#000207"
)

true_ax.set_xlim(
    -1,
    1
)

true_ax.set_ylim(
    -1,
    1
)

true_ax.set_aspect("equal")
true_ax.axis("off")
true_ax.set_visible(False)


# rings
final_ring_outer = Ellipse(
    (0, 0),
    1.52,
    0.56,
    angle=-18,
    facecolor="#7DC8E3",
    edgecolor="#D6F5FF",
    linewidth=1.6,
    alpha=0.30,
    zorder=4
)

final_ring_gap = Ellipse(
    (0, 0),
    0.93,
    0.33,
    angle=-18,
    facecolor="#000207",
    edgecolor="#78AFC4",
    linewidth=1.0,
    zorder=5
)

final_true_planet = Circle(
    (0, 0),
    0.31,
    facecolor="#3E7895",
    edgecolor="#DDF8FF",
    linewidth=1.5,
    zorder=8
)

final_true_highlight = Circle(
    (-0.07, 0.08),
    0.16,
    facecolor="#74B7D4",
    edgecolor="none",
    alpha=0.35,
    zorder=9
)

true_ax.add_patch(final_ring_outer)
true_ax.add_patch(final_ring_gap)
true_ax.add_patch(final_true_planet)
true_ax.add_patch(final_true_highlight)


# ============================================================
# FINAL SCENE — RINGLESS INTERPRETATION
# ============================================================

apparent_ax = fig.add_axes(
    [0.575, 0.32, 0.33, 0.43],
    zorder=80
)

apparent_ax.set_facecolor(
    "#000207"
)

apparent_ax.set_xlim(
    -1,
    1
)

apparent_ax.set_ylim(
    -1,
    1
)

apparent_ax.set_aspect("equal")
apparent_ax.axis("off")
apparent_ax.set_visible(False)


final_apparent_glow = Circle(
    (0, 0),
    0.63,
    facecolor="#5BBBE0",
    edgecolor="none",
    alpha=0.08,
    zorder=4
)

final_apparent_planet = Circle(
    (0, 0),
    0.58,
    facecolor="#477E99",
    edgecolor="#DDF8FF",
    linewidth=1.6,
    zorder=6
)

final_apparent_highlight = Circle(
    (-0.13, 0.15),
    0.30,
    facecolor="#73B4D0",
    edgecolor="none",
    alpha=0.30,
    zorder=7
)

apparent_ax.add_patch(
    final_apparent_glow
)

apparent_ax.add_patch(
    final_apparent_planet
)

apparent_ax.add_patch(
    final_apparent_highlight
)


# ============================================================
# FINAL SCENE — TEXT
# ============================================================

final_header = fig.text(
    0.5, 0.825,
    "THE SAME TRANSIT CAN IMPLY A VERY DIFFERENT PLANET",
    ha="center",
    va="center",
    color="white",
    fontsize=17,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_subheader = fig.text(
    0.5, 0.785,
    "if the occulting area includes unresolved rings",
    ha="center",
    va="center",
    color="#A7C9D9",
    fontsize=11,
    alpha=0,
    zorder=100
)


true_title_final = fig.text(
    0.26, 0.275,
    "PLANET + RINGS",
    ha="center",
    va="center",
    color="#CFF4FF",
    fontsize=14,
    fontweight="bold",
    alpha=0,
    zorder=100
)

true_note_final = fig.text(
    0.26, 0.230,
    "smaller true planetary radius",
    ha="center",
    va="center",
    color="#9ABBCB",
    fontsize=10,
    alpha=0,
    zorder=100
)


apparent_title_final = fig.text(
    0.74, 0.275,
    "RINGLESS MODEL",
    ha="center",
    va="center",
    color="#CFF4FF",
    fontsize=14,
    fontweight="bold",
    alpha=0,
    zorder=100
)

apparent_note_final = fig.text(
    0.74, 0.230,
    "larger inferred planetary radius",
    ha="center",
    va="center",
    color="#9ABBCB",
    fontsize=10,
    alpha=0,
    zorder=100
)


final_equation = fig.text(
    0.5, 0.155,
    "LARGER INFERRED RADIUS  →  LOWER INFERRED BULK DENSITY",
    ha="center",
    va="center",
    color="#AEEAFF",
    fontsize=12,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_caveat = fig.text(
    0.5, 0.112,
    "Possible explanation — not a detection of rings",
    ha="center",
    va="center",
    color="#7899A9",
    fontsize=9,
    alpha=0,
    zorder=100
)


# ============================================================
# CLOCK / PROGRESS DIAL
# Decorative only — NOT a physical timescale
# ============================================================

clock_ax = fig.add_axes(
    [0.030, 0.035, 0.080, 0.14],
    facecolor="none",
    zorder=300
)

clock_ax.set_xlim(-1, 1)
clock_ax.set_ylim(-1, 1)
clock_ax.set_aspect("equal")
clock_ax.axis("off")

dial_center = (0, 0)
dial_radius = 0.82


dial_outer = Circle(
    dial_center,
    dial_radius,
    facecolor="#07111A",
    edgecolor="#91DDF7",
    linewidth=1.3,
    alpha=0.80,
    zorder=30
)

dial_inner = Circle(
    dial_center,
    dial_radius * 0.82,
    facecolor="none",
    edgecolor="#52788C",
    linewidth=0.7,
    alpha=0.65,
    zorder=31
)

clock_ax.add_patch(dial_outer)
clock_ax.add_patch(dial_inner)


for i in range(12):

    a = 2 * np.pi * i / 12

    r1 = (
        dial_radius *
        (0.67 if i % 3 else 0.59)
    )

    r2 = (
        dial_radius *
        0.83
    )

    x1 = r1 * np.sin(a)
    y1 = r1 * np.cos(a)

    x2 = r2 * np.sin(a)
    y2 = r2 * np.cos(a)

    clock_ax.plot(
        [x1, x2],
        [y1, y2],
        color="#A7D8EA",
        lw=1.1 if i % 3 else 1.6,
        alpha=0.72,
        zorder=32
    )


hand_length = (
    dial_radius *
    0.56
)

dial_hand, = clock_ax.plot(
    [0, 0],
    [0, hand_length],
    color="#E8FAFF",
    lw=1.8,
    solid_capstyle="round",
    zorder=34
)

dial_pin = Circle(
    (0, 0),
    dial_radius * 0.065,
    facecolor="#E8FAFF",
    edgecolor="none",
    zorder=35
)

clock_ax.add_patch(
    dial_pin
)


# ============================================================
# UPDATE
# ============================================================

def update(frame):

    sec = frame / FPS

    t = frame / (
        NFRAMES - 1
    )


    # ========================================================
    # PHASE LABELS
    # ========================================================

    if sec < 4.0:

        phase_text.set_text(
            "TRUE SYSTEM"
        )

        phase_sub.set_text(
            "A planet surrounded by an inclined ring system"
        )

    elif sec < 8.0:

        phase_text.set_text(
            "TRANSIT"
        )

        phase_sub.set_text(
            "Planet and rings both block starlight"
        )

    else:

        phase_text.set_text(
            "RINGLESS INTERPRETATION"
        )

        phase_sub.set_text(
            "The total occulting area is assigned to the planet"
        )


    # ========================================================
    # TRANSIT MOTION
    # ========================================================

    transit_progress = smoothstep(
        (sec - 3.2) / 4.4
    )

    px = (
        -1.28 +
        2.56 *
        transit_progress
    )

    py = 0.05


    ring_outer.center = (
        px,
        py
    )

    ring_gap.center = (
        px,
        py
    )

    planet.center = (
        px,
        py
    )

    planet_highlight.center = (
        px - 0.035,
        py + 0.035
    )

    inferred_planet.center = (
        px,
        py
    )


    # ========================================================
    # TRANSIT DEPTH APPEARS
    # ========================================================

    depth_progress = smoothstep(
        (sec - 4.2) / 1.2
    )

    depth_line.set_alpha(
        depth_progress
    )

    depth_cap1.set_alpha(
        depth_progress
    )

    depth_cap2.set_alpha(
        depth_progress
    )

    depth_text.set_alpha(
        depth_progress
    )


    # ========================================================
    # RINGLESS MODEL
    #
    # Rings fade away while a larger circular planet is
    # inferred from the same occulting area.
    # ========================================================

    model_progress = smoothstep(
        (sec - 8.0) / 2.2
    )

    ring_outer.set_alpha(
        0.28 *
        (1.0 - model_progress)
    )

    ring_gap.set_alpha(
        1.0 - model_progress
    )

    inferred_planet.set_alpha(
        0.90 *
        model_progress
    )

    true_label.set_alpha(
        1.0 - model_progress
    )

    true_sub.set_alpha(
        1.0 - model_progress
    )

    inferred_label.set_alpha(
        model_progress
    )


    # ========================================================
    # INTERPRETATION CHAIN
    # ========================================================

    p1 = smoothstep(
        (sec - 8.3) / 0.8
    )

    p2 = smoothstep(
        (sec - 9.0) / 0.8
    )

    p3 = smoothstep(
        (sec - 9.7) / 0.8
    )

    p4 = smoothstep(
        (sec - 10.4) / 0.8
    )

    p5 = smoothstep(
        (sec - 11.1) / 0.8
    )


    info1.set_alpha(p1)
    arrow1.set_alpha(p2)
    info2.set_alpha(p3)
    arrow2.set_alpha(p4)
    info3.set_alpha(p5)


    # ========================================================
    # STAR SUBTLE PULSE
    # ========================================================

    pulse = (
        0.5 +
        0.5 *
        np.sin(
            2 *
            np.pi *
            sec / 3.1
        )
    )

    star_glow_1.set_alpha(
        0.045 +
        0.025 * pulse
    )

    star_glow_2.set_alpha(
        0.08 +
        0.04 * pulse
    )


    # ========================================================
    # FINAL SCENE TRANSITION
    #
    # 13.2 s -> transition begins
    # 14.0 s -> complete
    # 14–20 s -> six-second hold
    # ========================================================

    scene2 = smoothstep(
        (sec - 13.2) / 0.8
    )

    scene1_alpha = (
        1.0 -
        scene2
    )


    ax.set_alpha(
        scene1_alpha
    )

    curve_ax.set_alpha(
        scene1_alpha
    )

    info_ax.set_alpha(
        scene1_alpha
    )

    phase_text.set_alpha(
        scene1_alpha
    )

    phase_sub.set_alpha(
        scene1_alpha
    )

    main_subtitle.set_alpha(
        scene1_alpha
    )


    # ========================================================
    # FINAL BACKGROUND
    # ========================================================

    if scene2 > 0.005:

        result_bg.set_visible(
            True
        )

        result_bg.set_alpha(
            0.97 *
            scene2
        )

        true_ax.set_visible(
            True
        )

        true_ax.set_alpha(
            scene2
        )

        apparent_ax.set_visible(
            True
        )

        apparent_ax.set_alpha(
            scene2
        )

    else:

        result_bg.set_visible(
            False
        )

        true_ax.set_visible(
            False
        )

        apparent_ax.set_visible(
            False
        )


    # ========================================================
    # FINAL TEXT
    # ========================================================

    final_header.set_alpha(
        scene2
    )

    final_subheader.set_alpha(
        scene2
    )

    true_title_final.set_alpha(
        scene2
    )

    true_note_final.set_alpha(
        scene2
    )

    apparent_title_final.set_alpha(
        scene2
    )

    apparent_note_final.set_alpha(
        scene2
    )

    final_equation.set_alpha(
        scene2
    )

    final_caveat.set_alpha(
        scene2
    )


    # ========================================================
    # CLOCK
    # ========================================================

    dial_angle = (
        2.0 *
        np.pi *
        t
    )

    hand_x = (
        hand_length *
        np.sin(
            dial_angle
        )
    )

    hand_y = (
        hand_length *
        np.cos(
            dial_angle
        )
    )

    dial_hand.set_data(
        [0, hand_x],
        [0, hand_y]
    )


    return []


# ============================================================
# ANIMATION
# ============================================================

anim = FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000 / FPS,
    blit=False
)


# ============================================================
# EXPORT
# ============================================================

if OUTPUT == "webm":

    writer = FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2400,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-row-mt", "1",
            "-deadline", "good",
            "-cpu-used", "2"
        ]
    )

else:

    writer = FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=4000,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-preset", "medium",
            "-movflags", "+faststart"
        ]
    )


anim.save(
    OUTFILE,
    writer=writer,
    dpi=DPI
)

plt.close(fig)

print(f"Saved: {OUTFILE}")

display(
    Video(
        OUTFILE,
        embed=True,
        html_attributes="controls loop"
    )
)

# Экзопланета-пухляш — или планета с кольцами?

Планеты системы Kepler-51 относятся к числу самых необычных известных экзопланет. При массе всего в несколько раз больше земной их измеренные радиусы достигают примерно 6–10 радиусов Земли, а расчётная средняя плотность оказывается ниже 0,06 г/см³. За такие экстремальные характеристики подобные миры называют super-puffs — «планетами-пухляшами». 2609.25234v1.pdf

Но транзитный метод измеряет не физическую поверхность планеты напрямую. Он показывает, какая площадь звезды оказывается закрыта во время прохождения планеты перед её диском.

Если планету окружают достаточно крупные кольца, они тоже блокируют свет звезды. Если затем интерпретировать такой транзит при помощи модели без колец, вся закрытая площадь будет приписана самой планете. В результате её вычисленный радиус окажется больше настоящего.

А поскольку объём планеты зависит от куба радиуса, даже сравнительно небольшая ошибка в радиусе способна очень сильно изменить вычисленную среднюю плотность. Меньшая планета с кольцами может выглядеть как огромная сверхлёгкая планета-пухляш.

Авторы новой работы проверили эту гипотезу для системы Kepler-51. Их расчёты показывают, что существуют конфигурации колец, способные увеличить настоящую плотность планеты примерно в 2–5 раз по сравнению со стандартной интерпретацией транзита. Более экстремальные геометрические решения допускают ещё большие различия, хотя такие варианты значительно менее однозначны. 2609.25234v1.pdf

Это не означает, что кольца у планет Kepler-51 обнаружены. Работа проверяет, какие системы колец ещё совместимы с наблюдениями, а какие уже можно исключить. Таким образом, загадка необычайной «пухлости» этих миров пока остаётся открытой. 2609.25234v1.pdf

Анимация схематическая. Размеры планеты и колец, геометрия системы и форма кривой блеска иллюстрируют физический принцип и не являются реконструкцией конкретного транзита Kepler-51.

# Super-Puff Exoplanet — or a Ringed Planet?

The planets of the Kepler-51 system are among the most unusual exoplanets known. With masses of only several Earth masses, their measured radii reach approximately 6–10 Earth radii, producing inferred bulk densities below 0.06 g/cm³. Worlds with such extreme properties are known as super-puff planets. 2609.25234v1.pdf

But the transit method does not directly measure the physical surface of a planet. Instead, it measures how much of the stellar disk is obscured as the planet passes in front of its star.

If a planet is surrounded by sufficiently large rings, those rings also block starlight. When such a transit is interpreted using a model without rings, the entire occulting area can be attributed to the planet itself. The inferred planetary radius can therefore become substantially larger than its true radius.

Because planetary volume scales with the cube of radius, even a relatively modest error in radius can dramatically affect the inferred bulk density. A smaller ringed planet can therefore masquerade as an enormous, extremely low-density super-puff.

The authors of the new study tested this possibility for the Kepler-51 system. Their analysis finds ring configurations that could increase the planet’s true density by roughly a factor of 2–5 compared with the conventional ringless interpretation. More extreme geometrical solutions allow even larger differences, although these configurations are considerably more degenerate. 2609.25234v1.pdf

This does not mean that rings have been detected around the Kepler-51 planets. The study instead investigates which ring configurations remain compatible with the observations and which can already be excluded. The origin of these planets’ extraordinary apparent “puffiness” therefore remains an open question. 2609.25234v1.pdf

The animation is schematic. Planet and ring sizes, system geometry, and the transit light curve illustrate the physical principle and do not reconstruct a specific Kepler-51 transit.

---

# Part 2

In [ ]:
# ============================================================
# THE PHOTORING EFFECT
# Inspired by Zuluaga et al. (2026)
# "Probing Exoplanetary Rings with Asterodensity Profiling:
#  A PhotoRing Analysis of Kepler-51"
#
# One self-contained Jupyter cell
# Requires: numpy, matplotlib, ffmpeg
#
# STORY
#   0–4.5 s    Transit geometry
#   4.5–8 s    Transit observables -> stellar density
#   8–13.2 s   Compare transit-inferred and true stellar density
#   13.2–14 s  Transition
#   14–20 s    Kepler-51b vs Kepler-51d result
#
# IMPORTANT
# Transit curves and ring geometry are schematic.
# Ψ values and significance values are from the paper.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle, Ellipse, Rectangle
from IPython.display import Video, display


# ============================================================
# OUTPUT
# ============================================================

OUTPUT = "mp4"       # "webm" or "mp4"

FPS = 30
DURATION = 20
NFRAMES = FPS * DURATION
DPI = 120

if OUTPUT == "webm":
    OUTFILE = "25234_photoring_effect_kepler51.webm"
else:
    OUTFILE = "25234_photoring_effect_kepler51.mp4"


# ============================================================
# HELPERS
# ============================================================

def clamp(x, a=0.0, b=1.0):
    return np.minimum(np.maximum(x, a), b)


def smoothstep(x):
    x = clamp(x)
    return x * x * (3.0 - 2.0 * x)


BG = "#02050b"


# ============================================================
# FIGURE
# ============================================================

fig = plt.figure(
    figsize=(12.8, 7.2),
    facecolor=BG
)


# ============================================================
# GLOBAL TITLE
# ============================================================

main_title = fig.text(
    0.5, 0.955,
    "THE PHOTORING EFFECT",
    ha="center",
    va="center",
    color="white",
    fontsize=21,
    fontweight="bold",
    zorder=200
)

main_subtitle = fig.text(
    0.5, 0.916,
    "Can a transit reveal rings that we cannot directly resolve?",
    ha="center",
    va="center",
    color="#B7D6E8",
    fontsize=12,
    zorder=200
)


# ============================================================
# LEFT PANEL — TRANSITING SYSTEM
# ============================================================

ax = fig.add_axes(
    [0.035, 0.17, 0.45, 0.66],
    zorder=10
)

ax.set_facecolor(BG)

ax.set_xlim(
    -1.45,
    1.45
)

ax.set_ylim(
    -1.05,
    1.05
)

ax.set_aspect("equal")
ax.axis("off")


# ============================================================
# STAR
# ============================================================

star_glow = Circle(
    (0, 0),
    0.90,
    facecolor="#F2C15D",
    edgecolor="none",
    alpha=0.07,
    zorder=1
)

star = Circle(
    (0, 0),
    0.70,
    facecolor="#F2C05E",
    edgecolor="#FFE5A0",
    linewidth=1.3,
    alpha=0.97,
    zorder=2
)

star_inner = Circle(
    (-0.09, 0.07),
    0.56,
    facecolor="#FFD87B",
    edgecolor="none",
    alpha=0.40,
    zorder=3
)

ax.add_patch(star_glow)
ax.add_patch(star)
ax.add_patch(star_inner)


# ============================================================
# PLANET + RING
# ============================================================

planet_y = 0.08
planet_x = -1.18

planet = Circle(
    (planet_x, planet_y),
    0.12,
    facecolor="#396F8A",
    edgecolor="#D7F5FF",
    linewidth=1.1,
    zorder=15
)

ring_outer = Ellipse(
    (planet_x, planet_y),
    0.55,
    0.20,
    angle=-17,
    facecolor="#83CBE5",
    edgecolor="#D8F5FF",
    linewidth=1.1,
    alpha=0.30,
    zorder=12
)

ring_inner = Ellipse(
    (planet_x, planet_y),
    0.34,
    0.115,
    angle=-17,
    facecolor=BG,
    edgecolor="#759EAF",
    linewidth=0.7,
    zorder=13
)

ax.add_patch(ring_outer)
ax.add_patch(ring_inner)
ax.add_patch(planet)


# ============================================================
# TRANSIT PATH
# ============================================================

path_line, = ax.plot(
    [-1.25, 1.25],
    [planet_y, planet_y],
    color="#668899",
    lw=1.0,
    linestyle="--",
    alpha=0.28,
    zorder=5
)


# ============================================================
# TRANSIT CONTACT MARKERS
# ============================================================

contact_x = [
    -0.92,
    -0.57,
    0.57,
    0.92
]

contact_lines = []

for cx in contact_x:

    line, = ax.plot(
        [cx, cx],
        [-0.80, -0.68],
        color="#78DFFF",
        lw=1.4,
        alpha=0,
        zorder=25
    )

    contact_lines.append(
        line
    )


contact_texts = []

for i, cx in enumerate(contact_x):

    txt = ax.text(
        cx,
        -0.88,
        f"T{i+1}",
        ha="center",
        va="center",
        color="#9BE9FF",
        fontsize=9,
        fontweight="bold",
        alpha=0,
        zorder=25
    )

    contact_texts.append(
        txt
    )


# ============================================================
# LEFT LABEL
# ============================================================

geometry_title = ax.text(
    0,
    -0.94,
    "TRANSIT GEOMETRY",
    ha="center",
    va="center",
    color="#DDF5FF",
    fontsize=11,
    fontweight="bold",
    zorder=30
)


# ============================================================
# RIGHT — TRANSIT CURVE
# ============================================================

curve_ax = fig.add_axes(
    [0.545, 0.50, 0.405, 0.27],
    zorder=10
)

curve_ax.set_facecolor(
    "#06101A"
)

phase = np.linspace(
    -1.3,
    1.3,
    800
)

depth = 0.034
edge = 0.72
soft = 0.085

left_edge = (
    1 /
    (
        1 +
        np.exp(
            -(phase + edge) / soft
        )
    )
)

right_edge = (
    1 /
    (
        1 +
        np.exp(
            (phase - edge) / soft
        )
    )
)

shape = (
    left_edge *
    right_edge
)

flux = (
    1 -
    depth * shape
)


curve_line, = curve_ax.plot(
    phase,
    flux,
    color="#E5F9FF",
    lw=2.4
)

curve_ax.axhline(
    1,
    color="#567485",
    lw=0.8,
    alpha=0.45
)

curve_ax.set_xlim(
    -1.3,
    1.3
)

curve_ax.set_ylim(
    0.958,
    1.006
)

curve_ax.set_xlabel(
    "TRANSIT PHASE",
    color="#AFC8D7",
    fontsize=9,
    fontweight="bold"
)

curve_ax.set_ylabel(
    "FLUX",
    color="#AFC8D7",
    fontsize=9,
    fontweight="bold"
)

curve_ax.tick_params(
    colors="#94AEBE",
    labelsize=8
)

curve_ax.grid(
    color="#52738A",
    alpha=0.15
)

for spine in curve_ax.spines.values():
    spine.set_color("#52738A")

curve_ax.set_title(
    "TRANSIT OBSERVABLES",
    color="#DDF5FF",
    fontsize=12,
    fontweight="bold"
)


# ============================================================
# OBSERVABLE MARKERS
# ============================================================

obs_depth = curve_ax.text(
    0.96,
    0.982,
    "δ",
    ha="center",
    va="center",
    color="#8DE8FF",
    fontsize=15,
    fontweight="bold",
    alpha=0
)

obs_t14 = curve_ax.text(
    0,
    0.9615,
    "T14",
    ha="center",
    va="bottom",
    color="#8DE8FF",
    fontsize=10,
    fontweight="bold",
    alpha=0
)

t14_line, = curve_ax.plot(
    [-0.92, 0.92],
    [0.963, 0.963],
    color="#72DFFF",
    lw=1.5,
    alpha=0
)

obs_t23 = curve_ax.text(
    0,
    0.970,
    "T23",
    ha="center",
    va="bottom",
    color="#C0F2FF",
    fontsize=9,
    fontweight="bold",
    alpha=0
)

t23_line, = curve_ax.plot(
    [-0.57, 0.57],
    [0.9715, 0.9715],
    color="#A8EFFF",
    lw=1.3,
    alpha=0
)


# ============================================================
# RIGHT — ASTERODENSITY LOGIC
# ============================================================

logic_ax = fig.add_axes(
    [0.545, 0.14, 0.405, 0.27],
    zorder=10
)

logic_ax.set_facecolor(BG)
logic_ax.axis("off")


logic_obs = logic_ax.text(
    0.18, 0.76,
    "δ   T14   T23   P",
    ha="center",
    va="center",
    color="#DDF4FF",
    fontsize=13,
    fontweight="bold",
    alpha=0,
    transform=logic_ax.transAxes
)

logic_arrow1 = logic_ax.text(
    0.42, 0.76,
    "→",
    ha="center",
    va="center",
    color="#72DFFF",
    fontsize=18,
    fontweight="bold",
    alpha=0,
    transform=logic_ax.transAxes
)

logic_rho_obs = logic_ax.text(
    0.68, 0.76,
    "ρ★,obs",
    ha="center",
    va="center",
    color="#AEEAFF",
    fontsize=16,
    fontweight="bold",
    alpha=0,
    transform=logic_ax.transAxes
)

logic_caption = logic_ax.text(
    0.68, 0.60,
    "from transit geometry",
    ha="center",
    va="center",
    color="#819EAE",
    fontsize=9,
    alpha=0,
    transform=logic_ax.transAxes
)


logic_rho_true = logic_ax.text(
    0.68, 0.35,
    "ρ★,true",
    ha="center",
    va="center",
    color="#D8E6ED",
    fontsize=16,
    fontweight="bold",
    alpha=0,
    transform=logic_ax.transAxes
)

logic_true_caption = logic_ax.text(
    0.68, 0.20,
    "independent stellar estimate",
    ha="center",
    va="center",
    color="#819EAE",
    fontsize=9,
    alpha=0,
    transform=logic_ax.transAxes
)


logic_compare = logic_ax.text(
    0.22, 0.32,
    "COMPARE",
    ha="center",
    va="center",
    color="#DDF4FF",
    fontsize=11,
    fontweight="bold",
    alpha=0,
    transform=logic_ax.transAxes
)


# ============================================================
# PSI EQUATION
# ============================================================

psi_text = fig.text(
    0.75, 0.115,
    "Ψ = ρ★,obs / ρ★,true",
    ha="center",
    va="center",
    color="#A9EEFF",
    fontsize=14,
    fontweight="bold",
    alpha=0,
    zorder=40
)


# ============================================================
# PHASE TEXT
# ============================================================

phase_text = fig.text(
    0.26, 0.855,
    "",
    ha="center",
    va="center",
    color="white",
    fontsize=16,
    fontweight="bold",
    zorder=50
)

phase_sub = fig.text(
    0.26, 0.817,
    "",
    ha="center",
    va="center",
    color="#B7D3E7",
    fontsize=11,
    zorder=50
)


# ============================================================
# FINAL DARK BACKGROUND
# ============================================================

result_bg = fig.add_axes(
    [0.025, 0.075, 0.95, 0.80],
    zorder=70
)

result_bg.set_facecolor(
    "#000207"
)

result_bg.set_xticks([])
result_bg.set_yticks([])

for spine in result_bg.spines.values():
    spine.set_visible(False)

result_bg.set_visible(False)
result_bg.set_alpha(0)


# ============================================================
# FINAL RESULT PANELS
# ============================================================

b_ax = fig.add_axes(
    [0.09, 0.25, 0.37, 0.43],
    zorder=80
)

d_ax = fig.add_axes(
    [0.54, 0.25, 0.37, 0.43],
    zorder=80
)

for result_ax in [b_ax, d_ax]:

    result_ax.set_facecolor(
        "#030810"
    )

    result_ax.set_xlim(
        0.85,
        1.28
    )

    result_ax.set_ylim(
        0,
        1
    )

    result_ax.set_yticks([])

    result_ax.set_xticks(
        [0.9, 1.0, 1.1, 1.2]
    )

    result_ax.tick_params(
        axis="x",
        colors="#A9C2D0",
        labelsize=9
    )

    for spine in result_ax.spines.values():
        spine.set_color("#355061")

    result_ax.axvline(
        1.0,
        color="#C7D8E1",
        lw=1.2,
        linestyle="--",
        alpha=0.6
    )

    result_ax.text(
        1.0,
        0.12,
        "NO OFFSET",
        ha="center",
        va="center",
        color="#7895A5",
        fontsize=8
    )

    result_ax.set_xlabel(
        "Ψ = ρ★,obs / ρ★,true",
        color="#B9D0DC",
        fontsize=10,
        fontweight="bold"
    )


# Kepler-51b
b_ax.set_title(
    "KEPLER-51b",
    color="white",
    fontsize=15,
    fontweight="bold"
)

b_err = b_ax.errorbar(
    [1.18],
    [0.55],
    xerr=[0.06],
    fmt="o",
    markersize=10,
    capsize=6,
    color="#8BE7FF",
    ecolor="#8BE7FF",
    elinewidth=2.2
)

b_value = b_ax.text(
    1.18,
    0.77,
    "Ψ = 1.18 ± 0.06",
    ha="center",
    va="center",
    color="#BDF3FF",
    fontsize=12,
    fontweight="bold"
)

b_sig = b_ax.text(
    1.18,
    0.36,
    "2.9σ OFFSET",
    ha="center",
    va="center",
    color="#8BE7FF",
    fontsize=11,
    fontweight="bold"
)


# Kepler-51d
d_ax.set_title(
    "KEPLER-51d",
    color="white",
    fontsize=15,
    fontweight="bold"
)

d_err = d_ax.errorbar(
    [1.04],
    [0.55],
    xerr=[0.08],
    fmt="o",
    markersize=10,
    capsize=6,
    color="#D1E0E7",
    ecolor="#D1E0E7",
    elinewidth=2.2
)

d_value = d_ax.text(
    1.04,
    0.77,
    "Ψ = 1.04 ± 0.08",
    ha="center",
    va="center",
    color="#D9E8EF",
    fontsize=12,
    fontweight="bold"
)

d_sig = d_ax.text(
    1.04,
    0.36,
    "0.6σ OFFSET",
    ha="center",
    va="center",
    color="#A8BBC5",
    fontsize=11,
    fontweight="bold"
)


b_ax.set_visible(False)
d_ax.set_visible(False)


# ============================================================
# FINAL TEXT
# ============================================================

final_header = fig.text(
    0.5, 0.805,
    "SEARCHING FOR A PHOTORING OFFSET",
    ha="center",
    va="center",
    color="white",
    fontsize=18,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_sub = fig.text(
    0.5, 0.755,
    "Ψ = 1 means transit geometry agrees with the independent stellar density",
    ha="center",
    va="center",
    color="#A9C8D8",
    fontsize=10,
    alpha=0,
    zorder=100
)

final_b_note = fig.text(
    0.275, 0.190,
    "meaningful but sub-detection preference",
    ha="center",
    va="center",
    color="#8DB7C9",
    fontsize=9,
    alpha=0,
    zorder=100
)

final_d_note = fig.text(
    0.725, 0.190,
    "consistent with no anomaly",
    ha="center",
    va="center",
    color="#8FA4AF",
    fontsize=9,
    alpha=0,
    zorder=100
)

final_warning = fig.text(
    0.5, 0.125,
    "A PHOTORING OFFSET IS A DIAGNOSTIC — NOT A RING DETECTION",
    ha="center",
    va="center",
    color="#AEEAFF",
    fontsize=11,
    fontweight="bold",
    alpha=0,
    zorder=100
)


# ============================================================
# CLOCK / PROGRESS DIAL
# Decorative only
# ============================================================

clock_ax = fig.add_axes(
    [0.030, 0.035, 0.080, 0.14],
    facecolor="none",
    zorder=300
)

clock_ax.set_xlim(-1, 1)
clock_ax.set_ylim(-1, 1)
clock_ax.set_aspect("equal")
clock_ax.axis("off")

dial_radius = 0.82

dial_outer = Circle(
    (0, 0),
    dial_radius,
    facecolor="#07111A",
    edgecolor="#91DDF7",
    linewidth=1.3,
    alpha=0.80,
    zorder=30
)

dial_inner = Circle(
    (0, 0),
    dial_radius * 0.82,
    facecolor="none",
    edgecolor="#52788C",
    linewidth=0.7,
    alpha=0.65,
    zorder=31
)

clock_ax.add_patch(
    dial_outer
)

clock_ax.add_patch(
    dial_inner
)


for i in range(12):

    a = (
        2 *
        np.pi *
        i / 12
    )

    r1 = (
        dial_radius *
        (0.67 if i % 3 else 0.59)
    )

    r2 = (
        dial_radius *
        0.83
    )

    x1 = r1 * np.sin(a)
    y1 = r1 * np.cos(a)

    x2 = r2 * np.sin(a)
    y2 = r2 * np.cos(a)

    clock_ax.plot(
        [x1, x2],
        [y1, y2],
        color="#A7D8EA",
        lw=1.1 if i % 3 else 1.6,
        alpha=0.72,
        zorder=32
    )


hand_length = (
    dial_radius *
    0.56
)

dial_hand, = clock_ax.plot(
    [0, 0],
    [0, hand_length],
    color="#E8FAFF",
    lw=1.8,
    solid_capstyle="round",
    zorder=34
)

dial_pin = Circle(
    (0, 0),
    dial_radius * 0.065,
    facecolor="#E8FAFF",
    edgecolor="none",
    zorder=35
)

clock_ax.add_patch(
    dial_pin
)


# ============================================================
# UPDATE
# ============================================================

def update(frame):

    sec = frame / FPS
    t = frame / (NFRAMES - 1)


    # ========================================================
    # PHASE TEXT
    # ========================================================

    if sec < 4.5:

        phase_text.set_text(
            "OBSERVE THE TRANSIT"
        )

        phase_sub.set_text(
            "Measure how the occulting object crosses the star"
        )

    elif sec < 8.0:

        phase_text.set_text(
            "MEASURE THE GEOMETRY"
        )

        phase_sub.set_text(
            "Depth and contact times encode the transit shape"
        )

    else:

        phase_text.set_text(
            "ASTERODENSITY PROFILING"
        )

        phase_sub.set_text(
            "Compare transit-inferred and independently measured stellar density"
        )


    # ========================================================
    # PLANET MOTION
    # ========================================================

    transit_p = smoothstep(
        (sec - 0.8) / 6.2
    )

    px = (
        -1.18 +
        2.36 *
        transit_p
    )

    planet.center = (
        px,
        planet_y
    )

    ring_outer.center = (
        px,
        planet_y
    )

    ring_inner.center = (
        px,
        planet_y
    )


    # ========================================================
    # CONTACT MARKERS
    # ========================================================

    contact_p = smoothstep(
        (sec - 3.2) / 1.1
    )

    for line in contact_lines:
        line.set_alpha(
            0.75 * contact_p
        )

    for txt in contact_texts:
        txt.set_alpha(
            contact_p
        )


    # ========================================================
    # TRANSIT OBSERVABLES
    # ========================================================

    obs1 = smoothstep(
        (sec - 4.3) / 0.8
    )

    obs2 = smoothstep(
        (sec - 5.0) / 0.8
    )

    obs3 = smoothstep(
        (sec - 5.7) / 0.8
    )

    obs_depth.set_alpha(
        obs1
    )

    t14_line.set_alpha(
        obs2
    )

    obs_t14.set_alpha(
        obs2
    )

    t23_line.set_alpha(
        obs3
    )

    obs_t23.set_alpha(
        obs3
    )


    # ========================================================
    # ASTERODENSITY LOGIC
    # ========================================================

    p1 = smoothstep(
        (sec - 6.4) / 0.8
    )

    p2 = smoothstep(
        (sec - 7.0) / 0.7
    )

    p3 = smoothstep(
        (sec - 7.5) / 0.8
    )

    p4 = smoothstep(
        (sec - 8.5) / 0.8
    )

    p5 = smoothstep(
        (sec - 9.2) / 0.8
    )

    p6 = smoothstep(
        (sec - 10.1) / 0.8
    )


    logic_obs.set_alpha(
        p1
    )

    logic_arrow1.set_alpha(
        p2
    )

    logic_rho_obs.set_alpha(
        p3
    )

    logic_caption.set_alpha(
        p3
    )

    logic_rho_true.set_alpha(
        p4
    )

    logic_true_caption.set_alpha(
        p4
    )

    logic_compare.set_alpha(
        p5
    )

    psi_text.set_alpha(
        p6
    )


    # ========================================================
    # STAR PULSE
    # ========================================================

    pulse = (
        0.5 +
        0.5 *
        np.sin(
            2 *
            np.pi *
            sec / 3.2
        )
    )

    star_glow.set_alpha(
        0.055 +
        0.035 * pulse
    )


    # ========================================================
    # FINAL TRANSITION
    # ========================================================

    scene2 = smoothstep(
        (sec - 13.2) / 0.8
    )

    scene1_alpha = (
        1.0 -
        scene2
    )


    ax.set_alpha(
        scene1_alpha
    )

    curve_ax.set_alpha(
        scene1_alpha
    )

    logic_ax.set_alpha(
        scene1_alpha
    )

    phase_text.set_alpha(
        scene1_alpha
    )

    phase_sub.set_alpha(
        scene1_alpha
    )

    main_subtitle.set_alpha(
        scene1_alpha
    )

    psi_text.set_alpha(
        p6 * scene1_alpha
    )


    # ========================================================
    # FINAL BACKGROUND
    # ========================================================

    if scene2 > 0.005:

        result_bg.set_visible(
            True
        )

        result_bg.set_alpha(
            0.97 * scene2
        )

        b_ax.set_visible(
            True
        )

        b_ax.set_alpha(
            scene2
        )

        d_ax.set_visible(
            True
        )

        d_ax.set_alpha(
            scene2
        )

    else:

        result_bg.set_visible(
            False
        )

        b_ax.set_visible(
            False
        )

        d_ax.set_visible(
            False
        )


    # ========================================================
    # FINAL TEXT
    # ========================================================

    final_header.set_alpha(
        scene2
    )

    final_sub.set_alpha(
        scene2
    )

    final_b_note.set_alpha(
        scene2
    )

    final_d_note.set_alpha(
        scene2
    )

    final_warning.set_alpha(
        scene2
    )


    # ========================================================
    # CLOCK
    # ========================================================

    dial_angle = (
        2 *
        np.pi *
        t
    )

    hand_x = (
        hand_length *
        np.sin(
            dial_angle
        )
    )

    hand_y = (
        hand_length *
        np.cos(
            dial_angle
        )
    )

    dial_hand.set_data(
        [0, hand_x],
        [0, hand_y]
    )


    return []


# ============================================================
# ANIMATION
# ============================================================

anim = FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000 / FPS,
    blit=False
)


# ============================================================
# EXPORT
# ============================================================

if OUTPUT == "webm":

    writer = FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2400,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-row-mt", "1",
            "-deadline", "good",
            "-cpu-used", "2"
        ]
    )

else:

    writer = FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=4000,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-preset", "medium",
            "-movflags", "+faststart"
        ]
    )


anim.save(
    OUTFILE,
    writer=writer,
    dpi=DPI
)

plt.close(fig)

print(
    f"Saved: {OUTFILE}"
)

display(
    Video(
        OUTFILE,
        embed=True,
        html_attributes="controls loop"
    )
)

# Эффект PhotoRing: как искать невидимые кольца экзопланет

Даже если кольца экзопланеты невозможно непосредственно различить на изображении, они могут оставить след в геометрии её транзита. Метод PhotoRing использует разновидность asterodensity profiling: из глубины транзита, его полной продолжительности, времени между внутренними контактами и орбитального периода можно определить плотность звезды, которую предполагает наблюдаемая геометрия транзита. 2609.25234v1.pdf

Эту величину ρ★,obs затем сравнивают с настоящей плотностью звезды ρ★,true, полученной независимыми методами. Авторы характеризуют различие отношением

Ψ = ρ★,obs / ρ★,true.

Если Ψ ≈ 1, транзит согласуется с обычной моделью. Неучтённые кольца способны изменить длительность и геометрию транзита и тем самым вызвать PhotoRing-смещение.

Для Kepler-51b авторы получили Ψ = 1.18 ± 0.06 — отклонение от единицы на 2.9σ. Для Kepler-51d результат составляет Ψ = 1.04 ± 0.08, всего 0.6σ, то есть полностью совместим с отсутствием аномалии. 2609.25234v1.pdf

Однако даже результат Kepler-51b пока не является обнаружением колец. Его статистическая значимость зависит от независимо определённой плотности звезды, а 30-минутная временная дискретизация данных Kepler плохо разрешает особенно важные участки входа и выхода планеты из транзита. Авторы рассматривают результат как демонстрацию метода и способ ограничить возможную геометрию колец. 2609.25234v1.pdf

Форма транзита и изображение системы в анимации схематические. Численные значения Ψ и статистической значимости соответствуют результатам статьи.

# The PhotoRing Effect: Searching for Invisible Exoplanet Rings

Even when an exoplanetary ring system cannot be spatially resolved, rings can leave a signature in the geometry of a planetary transit. The PhotoRing technique uses a form of asterodensity profiling: transit depth, total duration, the interval between the inner contacts, and orbital period can be used to infer the stellar density implied by the observed transit geometry. 2609.25234v1.pdf

This transit-derived value, ρ★,obs, can then be compared with the independently determined true stellar density, ρ★,true. The study characterizes the difference using

Ψ = ρ★,obs / ρ★,true.

When Ψ ≈ 1, the transit geometry is consistent with the conventional interpretation. Unmodelled rings can alter the duration and geometry of the transit and produce a measurable PhotoRing offset.

For Kepler-51b, the authors measure Ψ = 1.18 ± 0.06, corresponding to a 2.9σ offset from unity. For Kepler-51d, they obtain Ψ = 1.04 ± 0.08, only a 0.6σ offset and therefore consistent with no anomaly. 2609.25234v1.pdf

The Kepler-51b result is nevertheless not a detection of rings. Its significance depends on the independently determined stellar density, while Kepler’s approximately 30-minute cadence limits the ability to resolve the crucial ingress and egress morphology. The authors therefore present the result as a demonstration of the PhotoRing method and as a way of constraining possible ring geometries. 2609.25234v1.pdf

The transit profile and system geometry shown in the animation are schematic. The numerical Ψ values and statistical significances are taken from the study.

---

# Part 3

In [ ]:
# ============================================================
# HOW PUFFY IS KEPLER-51b REALLY?
# Inspired by Zuluaga et al. (2026)
# "Probing Exoplanetary Rings with Asterodensity Profiling:
#  A PhotoRing Analysis of Kepler-51"
#
# One self-contained Jupyter cell
# Requires: numpy, matplotlib, ffmpeg
#
# STORY
#   0–4.5 s    Ringless interpretation: huge super-puff
#   4.5–9 s    Separate planet radius from ring occulting area
#   9–13.2 s   Smaller radius -> higher true bulk density
#   13.2–14 s  Transition
#   14–20 s    Final density landscape: ×2–5 and ×15–100
#
# IMPORTANT
# This animation illustrates the inference concept.
# It is NOT a physical contraction of the planet.
# Planet/ring geometry is schematic.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle, Ellipse, Rectangle
from IPython.display import Video, display


# ============================================================
# OUTPUT
# ============================================================

OUTPUT = "mp4"       # "webm" or "mp4"

FPS = 30
DURATION = 20
NFRAMES = FPS * DURATION
DPI = 120

if OUTPUT == "webm":
    OUTFILE = "25234_how_puffy_is_kepler51b.webm"
else:
    OUTFILE = "25234_how_puffy_is_kepler51b.mp4"


# ============================================================
# HELPERS
# ============================================================

def clamp(x, a=0.0, b=1.0):
    return np.minimum(np.maximum(x, a), b)


def smoothstep(x):
    x = clamp(x)
    return x * x * (3.0 - 2.0 * x)


BG = "#02050b"


# ============================================================
# FIGURE
# ============================================================

fig = plt.figure(
    figsize=(12.8, 7.2),
    facecolor=BG
)


# ============================================================
# GLOBAL TITLE
# ============================================================

main_title = fig.text(
    0.5, 0.955,
    "HOW PUFFY IS KEPLER-51b REALLY?",
    ha="center",
    va="center",
    color="white",
    fontsize=21,
    fontweight="bold",
    zorder=200
)

main_subtitle = fig.text(
    0.5, 0.916,
    "Rings can change the radius — and therefore the density — inferred from a transit",
    ha="center",
    va="center",
    color="#B7D6E8",
    fontsize=11,
    zorder=200
)


# ============================================================
# LEFT PANEL — PLANET INTERPRETATION
# ============================================================

planet_ax = fig.add_axes(
    [0.045, 0.18, 0.46, 0.64],
    zorder=10
)

planet_ax.set_facecolor(BG)

planet_ax.set_xlim(
    -1.25,
    1.25
)

planet_ax.set_ylim(
    -1.10,
    1.10
)

planet_ax.set_aspect("equal")
planet_ax.axis("off")


# ============================================================
# APPARENT SUPER-PUFF
# ============================================================

APPARENT_R = 0.62
TRUE_R = 0.33


apparent_glow = Circle(
    (0, 0.08),
    APPARENT_R * 1.12,
    facecolor="#58B5D9",
    edgecolor="none",
    alpha=0.08,
    zorder=2
)

apparent_planet = Circle(
    (0, 0.08),
    APPARENT_R,
    facecolor="#467F9D",
    edgecolor="#D9F7FF",
    linewidth=1.8,
    alpha=0.96,
    zorder=4
)

apparent_highlight = Circle(
    (-0.15, 0.23),
    APPARENT_R * 0.48,
    facecolor="#73B7D3",
    edgecolor="none",
    alpha=0.30,
    zorder=5
)

planet_ax.add_patch(
    apparent_glow
)

planet_ax.add_patch(
    apparent_planet
)

planet_ax.add_patch(
    apparent_highlight
)


# ============================================================
# TRUE SMALLER PLANET + RINGS
# ============================================================

ring_outer = Ellipse(
    (0, 0.08),
    1.55,
    0.56,
    angle=-18,
    facecolor="#82CBE5",
    edgecolor="#D7F5FF",
    linewidth=1.5,
    alpha=0,
    zorder=8
)

ring_inner = Ellipse(
    (0, 0.08),
    0.91,
    0.31,
    angle=-18,
    facecolor=BG,
    edgecolor="#79A7BA",
    linewidth=0.9,
    alpha=0,
    zorder=9
)

true_planet = Circle(
    (0, 0.08),
    TRUE_R,
    facecolor="#386E89",
    edgecolor="#DDF8FF",
    linewidth=1.5,
    alpha=0,
    zorder=11
)

true_highlight = Circle(
    (-0.075, 0.16),
    TRUE_R * 0.50,
    facecolor="#73B7D3",
    edgecolor="none",
    alpha=0,
    zorder=12
)

planet_ax.add_patch(
    ring_outer
)

planet_ax.add_patch(
    ring_inner
)

planet_ax.add_patch(
    true_planet
)

planet_ax.add_patch(
    true_highlight
)


# ============================================================
# SIZE GUIDES
# ============================================================

radius_line, = planet_ax.plot(
    [0, APPARENT_R],
    [0.08, 0.08],
    color="#9CEAFF",
    lw=2.0,
    alpha=0.8,
    zorder=20
)

radius_tick, = planet_ax.plot(
    [APPARENT_R, APPARENT_R],
    [0.03, 0.13],
    color="#9CEAFF",
    lw=1.5,
    alpha=0.8,
    zorder=20
)

radius_text = planet_ax.text(
    0.31,
    -0.02,
    "INFERRED RADIUS",
    ha="center",
    va="center",
    color="#B9F1FF",
    fontsize=10,
    fontweight="bold",
    zorder=20
)


# ============================================================
# PLANET LABEL
# ============================================================

planet_label = planet_ax.text(
    0,
    -0.78,
    "RINGLESS MODEL",
    ha="center",
    va="center",
    color="white",
    fontsize=14,
    fontweight="bold",
    zorder=30
)

planet_sub = planet_ax.text(
    0,
    -0.92,
    "all occulting area assigned to the planet",
    ha="center",
    va="center",
    color="#93B1C0",
    fontsize=9,
    zorder=30
)


# ============================================================
# RIGHT PANEL — DENSITY RELATION
# ============================================================

density_ax = fig.add_axes(
    [0.56, 0.22, 0.38, 0.54],
    zorder=10
)

density_ax.set_facecolor(
    "#06101A"
)

density_ax.set_xlim(
    0,
    1
)

density_ax.set_ylim(
    0,
    1
)

density_ax.set_xticks([])
density_ax.set_yticks([])

for spine in density_ax.spines.values():
    spine.set_color("#426071")


density_ax.set_title(
    "WHY RADIUS MATTERS",
    color="#DDF5FF",
    fontsize=13,
    fontweight="bold",
    pad=13
)


mass_text = density_ax.text(
    0.5, 0.83,
    "MASS",
    ha="center",
    va="center",
    color="#D9EDF7",
    fontsize=13,
    fontweight="bold"
)

mass_sub = density_ax.text(
    0.5, 0.75,
    "constrained independently",
    ha="center",
    va="center",
    color="#819EAE",
    fontsize=9
)


plus_text = density_ax.text(
    0.5, 0.64,
    "+",
    ha="center",
    va="center",
    color="#72DFFF",
    fontsize=20,
    fontweight="bold"
)


radius_logic = density_ax.text(
    0.5, 0.54,
    "PLANETARY RADIUS",
    ha="center",
    va="center",
    color="#D9EDF7",
    fontsize=13,
    fontweight="bold"
)


arrow_logic = density_ax.text(
    0.5, 0.42,
    "↓",
    ha="center",
    va="center",
    color="#72DFFF",
    fontsize=20,
    fontweight="bold"
)


density_logic = density_ax.text(
    0.5, 0.31,
    "BULK DENSITY",
    ha="center",
    va="center",
    color="#AEEAFF",
    fontsize=14,
    fontweight="bold"
)


formula = density_ax.text(
    0.5, 0.17,
    "ρ ∝ M / R³",
    ha="center",
    va="center",
    color="#C5F3FF",
    fontsize=17,
    fontweight="bold"
)


# ============================================================
# DENSITY FACTOR DISPLAY
# ============================================================

factor_box = fig.text(
    0.75, 0.145,
    "DENSITY ×1",
    ha="center",
    va="center",
    color="#DDF5FF",
    fontsize=15,
    fontweight="bold",
    bbox=dict(
        boxstyle="round,pad=0.55",
        facecolor="#07131E",
        edgecolor="#52788C",
        linewidth=1.1
    ),
    zorder=40
)

factor_patch = factor_box.get_bbox_patch()


# ============================================================
# PHASE TEXT
# ============================================================

phase_text = fig.text(
    0.275, 0.855,
    "",
    ha="center",
    va="center",
    color="white",
    fontsize=16,
    fontweight="bold",
    zorder=50
)

phase_sub = fig.text(
    0.275, 0.817,
    "",
    ha="center",
    va="center",
    color="#B7D3E7",
    fontsize=11,
    zorder=50
)


# ============================================================
# FINAL BACKGROUND
# ============================================================

result_bg = fig.add_axes(
    [0.025, 0.075, 0.95, 0.80],
    zorder=70
)

result_bg.set_facecolor(
    "#000207"
)

result_bg.set_xticks([])
result_bg.set_yticks([])

for spine in result_bg.spines.values():
    spine.set_visible(False)

result_bg.set_visible(False)
result_bg.set_alpha(0)


# ============================================================
# FINAL — PLANET + RINGS
# ============================================================

final_planet_ax = fig.add_axes(
    [0.07, 0.29, 0.31, 0.43],
    zorder=80
)

final_planet_ax.set_facecolor(
    "#000207"
)

final_planet_ax.set_xlim(
    -1,
    1
)

final_planet_ax.set_ylim(
    -1,
    1
)

final_planet_ax.set_aspect("equal")
final_planet_ax.axis("off")
final_planet_ax.set_visible(False)


fp_ring_outer = Ellipse(
    (0, 0),
    1.60,
    0.58,
    angle=-18,
    facecolor="#7FC9E3",
    edgecolor="#D7F5FF",
    linewidth=1.5,
    alpha=0.30,
    zorder=5
)

fp_ring_inner = Ellipse(
    (0, 0),
    0.94,
    0.32,
    angle=-18,
    facecolor="#000207",
    edgecolor="#789FB0",
    linewidth=0.9,
    zorder=6
)

fp_planet = Circle(
    (0, 0),
    0.34,
    facecolor="#3D7692",
    edgecolor="#DDF8FF",
    linewidth=1.5,
    zorder=8
)

fp_highlight = Circle(
    (-0.08, 0.09),
    0.17,
    facecolor="#75B7D3",
    edgecolor="none",
    alpha=0.32,
    zorder=9
)

final_planet_ax.add_patch(
    fp_ring_outer
)

final_planet_ax.add_patch(
    fp_ring_inner
)

final_planet_ax.add_patch(
    fp_planet
)

final_planet_ax.add_patch(
    fp_highlight
)


# ============================================================
# FINAL — DENSITY SCALE
# ============================================================

scale_ax = fig.add_axes(
    [0.43, 0.32, 0.49, 0.33],
    zorder=80
)

scale_ax.set_facecolor(
    "#030810"
)

scale_ax.set_xlim(
    0,
    2.08
)

scale_ax.set_ylim(
    0,
    1
)

scale_ax.set_yticks([])

scale_ax.set_xticks(
    [
        0,
        np.log10(2),
        np.log10(5),
        1,
        np.log10(15),
        2
    ]
)

scale_ax.set_xticklabels(
    [
        "×1",
        "×2",
        "×5",
        "×10",
        "×15",
        "×100"
    ],
    color="#AEC6D3",
    fontsize=9
)

scale_ax.tick_params(
    axis="x",
    colors="#AEC6D3"
)

for spine in scale_ax.spines.values():
    spine.set_color("#355061")


scale_ax.set_xlabel(
    "TRUE DENSITY RELATIVE TO THE RINGLESS INTERPRETATION",
    color="#B9D1DC",
    fontsize=9,
    fontweight="bold",
    labelpad=12
)


# baseline
scale_ax.plot(
    [0, 2],
    [0.50, 0.50],
    color="#607C8C",
    lw=2,
    zorder=2
)


# preferred / compact-ring region: 2–5
preferred_rect = Rectangle(
    (
        np.log10(2),
        0.34
    ),
    np.log10(5) - np.log10(2),
    0.32,
    facecolor="#58B9D8",
    edgecolor="#AEEAFF",
    linewidth=1.3,
    alpha=0.52,
    zorder=4
)

scale_ax.add_patch(
    preferred_rect
)


preferred_text = scale_ax.text(
    (
        np.log10(2) +
        np.log10(5)
    ) / 2,
    0.76,
    "COMPACT-RING\nSOLUTIONS",
    ha="center",
    va="center",
    color="#BDF3FF",
    fontsize=9,
    fontweight="bold",
    zorder=10
)


# extreme multimodal region: 15–100
extreme_rect = Rectangle(
    (
        np.log10(15),
        0.38
    ),
    2 - np.log10(15),
    0.24,
    facecolor="#52778A",
    edgecolor="#88AABA",
    linewidth=1.1,
    linestyle="--",
    alpha=0.30,
    zorder=3
)

scale_ax.add_patch(
    extreme_rect
)


extreme_text = scale_ax.text(
    (
        np.log10(15) + 2
    ) / 2,
    0.76,
    "OTHER ALLOWED\nGEOMETRIES",
    ha="center",
    va="center",
    color="#9DB7C4",
    fontsize=8,
    fontweight="bold",
    zorder=10
)


# ============================================================
# FINAL TEXT
# ============================================================

final_header = fig.text(
    0.5, 0.815,
    "RINGS CAN CHANGE THE INFERRED PLANET",
    ha="center",
    va="center",
    color="white",
    fontsize=18,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_subheader = fig.text(
    0.5, 0.770,
    "The same transit can be compatible with a smaller, denser planet",
    ha="center",
    va="center",
    color="#A7C9D9",
    fontsize=11,
    alpha=0,
    zorder=100
)


final_planet_title = fig.text(
    0.225, 0.255,
    "PUFF RINGED PLANET",
    ha="center",
    va="center",
    color="#CFF4FF",
    fontsize=13,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_planet_note = fig.text(
    0.225, 0.215,
    "planet + unresolved ring contribution",
    ha="center",
    va="center",
    color="#91AFBE",
    fontsize=9,
    alpha=0,
    zorder=100
)


final_main_result = fig.text(
    0.675, 0.250,
    "TRUE DENSITY ≈ 2–5× HIGHER",
    ha="center",
    va="center",
    color="#AEEAFF",
    fontsize=13,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_extreme_note = fig.text(
    0.675, 0.205,
    "Some degenerate solutions reach ≈15–100×",
    ha="center",
    va="center",
    color="#8FAAB8",
    fontsize=9,
    alpha=0,
    zorder=100
)


final_warning = fig.text(
    0.5, 0.125,
    "ALLOWED RING SOLUTIONS — NOT A MEASUREMENT OF THE TRUE PLANETARY DENSITY",
    ha="center",
    va="center",
    color="#9EC8D9",
    fontsize=9,
    fontweight="bold",
    alpha=0,
    zorder=100
)


# ============================================================
# CLOCK / PROGRESS DIAL
# Decorative only — not a physical timescale
# ============================================================

clock_ax = fig.add_axes(
    [0.030, 0.035, 0.080, 0.14],
    facecolor="none",
    zorder=300
)

clock_ax.set_xlim(
    -1,
    1
)

clock_ax.set_ylim(
    -1,
    1
)

clock_ax.set_aspect("equal")
clock_ax.axis("off")

dial_radius = 0.82

dial_outer = Circle(
    (0, 0),
    dial_radius,
    facecolor="#07111A",
    edgecolor="#91DDF7",
    linewidth=1.3,
    alpha=0.80,
    zorder=30
)

dial_inner = Circle(
    (0, 0),
    dial_radius * 0.82,
    facecolor="none",
    edgecolor="#52788C",
    linewidth=0.7,
    alpha=0.65,
    zorder=31
)

clock_ax.add_patch(
    dial_outer
)

clock_ax.add_patch(
    dial_inner
)


for i in range(12):

    a = (
        2 *
        np.pi *
        i / 12
    )

    r1 = (
        dial_radius *
        (0.67 if i % 3 else 0.59)
    )

    r2 = (
        dial_radius *
        0.83
    )

    x1 = r1 * np.sin(a)
    y1 = r1 * np.cos(a)

    x2 = r2 * np.sin(a)
    y2 = r2 * np.cos(a)

    clock_ax.plot(
        [x1, x2],
        [y1, y2],
        color="#A7D8EA",
        lw=1.1 if i % 3 else 1.6,
        alpha=0.72,
        zorder=32
    )


hand_length = (
    dial_radius *
    0.56
)

dial_hand, = clock_ax.plot(
    [0, 0],
    [0, hand_length],
    color="#E8FAFF",
    lw=1.8,
    solid_capstyle="round",
    zorder=34
)

dial_pin = Circle(
    (0, 0),
    dial_radius * 0.065,
    facecolor="#E8FAFF",
    edgecolor="none",
    zorder=35
)

clock_ax.add_patch(
    dial_pin
)


# ============================================================
# UPDATE
# ============================================================

def update(frame):

    sec = frame / FPS
    t = frame / (NFRAMES - 1)


    # ========================================================
    # PHASE LABEL
    # ========================================================

    if sec < 4.5:

        phase_text.set_text(
            "THE SUPER-PUFF INTERPRETATION"
        )

        phase_sub.set_text(
            "The entire transit depth is assigned to one enormous planet"
        )

    elif sec < 9.0:

        phase_text.set_text(
            "ADD UNRESOLVED RINGS"
        )

        phase_sub.set_text(
            "Part of the occulting area may belong to the rings"
        )

    else:

        phase_text.set_text(
            "REVISE THE BULK DENSITY"
        )

        phase_sub.set_text(
            "A smaller planetary radius means a denser planet"
        )


    # ========================================================
    # MORPH APPARENT PLANET -> TRUE PLANET + RINGS
    # ========================================================

    morph = smoothstep(
        (sec - 4.5) / 3.0
    )


    # apparent planet shrinks away
    apparent_planet.set_radius(
        APPARENT_R -
        (
            APPARENT_R - TRUE_R
        ) * morph
    )

    apparent_glow.set_radius(
        APPARENT_R * 1.12 -
        (
            APPARENT_R * 1.12 -
            TRUE_R * 1.12
        ) * morph
    )

    apparent_highlight.set_radius(
        APPARENT_R * 0.48 -
        (
            APPARENT_R * 0.48 -
            TRUE_R * 0.48
        ) * morph
    )

    apparent_highlight.center = (
        -0.15 +
        0.075 * morph,
        0.23 -
        0.07 * morph
    )


    # rings emerge
    ring_outer.set_alpha(
        0.30 * morph
    )

    ring_inner.set_alpha(
        morph
    )

    true_planet.set_alpha(
        morph
    )

    true_highlight.set_alpha(
        0.32 * morph
    )


    # apparent disk fades as true planet becomes dominant
    apparent_planet.set_alpha(
        0.96 * (1.0 - morph)
    )

    apparent_glow.set_alpha(
        0.08 * (1.0 - morph)
    )

    apparent_highlight.set_alpha(
        0.30 * (1.0 - morph)
    )


    # ========================================================
    # RADIUS INDICATOR SHRINKS
    # ========================================================

    current_r = (
        APPARENT_R -
        (
            APPARENT_R - TRUE_R
        ) * morph
    )

    radius_line.set_data(
        [0, current_r],
        [0.08, 0.08]
    )

    radius_tick.set_data(
        [current_r, current_r],
        [0.03, 0.13]
    )

    radius_text.set_position(
        (
            current_r / 2,
            -0.02
        )
    )


    # ========================================================
    # LABEL CHANGE
    # ========================================================

    if morph < 0.5:

        planet_label.set_text(
            "RINGLESS MODEL"
        )

        planet_sub.set_text(
            "all occulting area assigned to the planet"
        )

    else:

        planet_label.set_text(
            "PLANET + RINGS"
        )

        planet_sub.set_text(
            "the planetary disk can be smaller"
        )


    # ========================================================
    # DENSITY FACTOR
    #
    # Animate only to the conservative 2–5× region.
    # This is an inference illustration, not physical evolution.
    # ========================================================

    density_progress = smoothstep(
        (sec - 8.0) / 3.0
    )

    displayed_factor = (
        1 +
        3.0 *
        density_progress
    )

    if displayed_factor < 1.05:

        factor_box.set_text(
            "DENSITY ×1"
        )

    else:

        factor_box.set_text(
            f"DENSITY ×{displayed_factor:.1f}"
        )


    factor_box.set_color(
        "#DDF5FF"
        if density_progress < 0.15
        else "#AEEAFF"
    )

    factor_patch.set_edgecolor(
        "#52788C"
        if density_progress < 0.15
        else "#76DDFB"
    )


    # ========================================================
    # SUBTLE PLANET PULSE
    # ========================================================

    pulse = (
        0.5 +
        0.5 *
        np.sin(
            2 *
            np.pi *
            sec / 3.2
        )
    )

    if morph < 0.95:

        apparent_glow.set_alpha(
            0.08 *
            (1.0 - morph) *
            (
                0.75 +
                0.25 * pulse
            )
        )


    # ========================================================
    # FINAL TRANSITION
    # ========================================================

    scene2 = smoothstep(
        (sec - 13.2) / 0.8
    )

    scene1_alpha = (
        1.0 -
        scene2
    )


    planet_ax.set_alpha(
        scene1_alpha
    )

    density_ax.set_alpha(
        scene1_alpha
    )

    phase_text.set_alpha(
        scene1_alpha
    )

    phase_sub.set_alpha(
        scene1_alpha
    )

    main_subtitle.set_alpha(
        scene1_alpha
    )

    factor_box.set_alpha(
        scene1_alpha
    )

    factor_patch.set_alpha(
        scene1_alpha
    )


    # Fade global title for clean final card
    main_title.set_alpha(
        scene1_alpha
    )


    # ========================================================
    # FINAL BACKGROUND
    # ========================================================

    if scene2 > 0.005:

        result_bg.set_visible(
            True
        )

        result_bg.set_alpha(
            0.98 * scene2
        )

        final_planet_ax.set_visible(
            True
        )

        final_planet_ax.set_alpha(
            scene2
        )

        scale_ax.set_visible(
            True
        )

        scale_ax.set_alpha(
            scene2
        )

    else:

        result_bg.set_visible(
            False
        )

        final_planet_ax.set_visible(
            False
        )

        scale_ax.set_visible(
            False
        )


    # ========================================================
    # FINAL TEXT
    # ========================================================

    final_header.set_alpha(
        scene2
    )

    final_subheader.set_alpha(
        scene2
    )

    final_planet_title.set_alpha(
        scene2
    )

    final_planet_note.set_alpha(
        scene2
    )

    final_main_result.set_alpha(
        scene2
    )

    final_extreme_note.set_alpha(
        scene2
    )

    final_warning.set_alpha(
        scene2
    )


    # ========================================================
    # CLOCK
    # ========================================================

    dial_angle = (
        2 *
        np.pi *
        t
    )

    hand_x = (
        hand_length *
        np.sin(
            dial_angle
        )
    )

    hand_y = (
        hand_length *
        np.cos(
            dial_angle
        )
    )

    dial_hand.set_data(
        [0, hand_x],
        [0, hand_y]
    )


    return []


# ============================================================
# ANIMATION
# ============================================================

anim = FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000 / FPS,
    blit=False
)


# ============================================================
# EXPORT
# ============================================================

if OUTPUT == "webm":

    writer = FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2400,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-row-mt", "1",
            "-deadline", "good",
            "-cpu-used", "2"
        ]
    )

else:

    writer = FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=4000,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-preset", "medium",
            "-movflags", "+faststart"
        ]
    )


anim.save(
    OUTFILE,
    writer=writer,
    dpi=DPI
)

plt.close(fig)

print(
    f"Saved: {OUTFILE}"
)

display(
    Video(
        OUTFILE,
        embed=True,
        html_attributes="controls loop"
    )
)

# Насколько на самом деле «пухлая» Kepler-51b?

Планеты системы Kepler-51 выглядят почти неправдоподобно разреженными: при массе всего в несколько масс Земли их транзитные радиусы сравнимы с радиусами гораздо более массивных газовых планет. Но вычисленная плотность критически зависит от того, правильно ли определён радиус.

Если часть света звезды во время транзита закрывает не сама планета, а окружающие её кольца, стандартная модель без колец завышает радиус планеты. При той же известной массе меньший настоящий радиус означает значительно более высокую среднюю плотность.

В моделях, исследованных авторами для Kepler-51, компактные умеренно наклонённые кольца допускают решения, при которых настоящая плотность планеты оказывается примерно в 2–5 раз выше, чем следует из обычной модели без колец. 2609.25234v1.pdf

Пространство решений, однако, сильно вырождено. Авторы также находят допустимые конфигурации с гораздо меньшими планетарными радиусами, в которых плотность возрастает примерно в 15–100 раз. Такие варианты могут переместить объект в область плотностей обычных богатых летучими веществами или даже каменистых планет. Авторы подчёркивают, что это мультимодальный набор геометрических решений, а не измерение настоящей плотности Kepler-51b. 2609.25234v1.pdf

Поэтому главный результат здесь не в том, что Kepler-51b «на самом деле плотная». Он в другом: по одному транзитному радиусу мы не обязательно знаем физический размер самой планеты, если вокруг неё существует неразрешённая система колец.

Анимация показывает изменение интерпретации одних и тех же наблюдений, а не физическое сжатие планеты. Геометрия планеты и колец схематическая.

# How Puffy Is Kepler-51b Really?

The planets of the Kepler-51 system appear extraordinarily diffuse: despite having masses of only several Earth masses, their transit-derived radii are comparable to those of much more massive gas planets. But the inferred bulk density depends critically on whether the planetary radius has been measured correctly.

If some of the stellar light is blocked by rings rather than by the planet itself, a conventional ringless transit model overestimates the planetary radius. For the same independently constrained mass, a smaller true radius implies a substantially higher bulk density.

Among the configurations explored for Kepler-51, compact and moderately inclined rings produce solutions in which the true planetary density can be roughly 2–5 times higher than inferred from a conventional ringless model. 2609.25234v1.pdf

The solution space is highly degenerate, however. The authors also find viable configurations with substantially smaller planetary radii, increasing the inferred density by approximately 15–100 times. Some of these solutions would place the planet within the density range of ordinary volatile-rich or even rocky planets. These represent a multimodal set of possible geometries, not a measurement of Kepler-51b’s actual density. 2609.25234v1.pdf

The central result is therefore not that Kepler-51b has been shown to be a dense planet. Rather, a transit-derived radius does not necessarily equal the physical radius of the planet itself when unresolved rings may contribute to the occulting area.

The animation represents a change in the interpretation of the same observations, not the physical contraction of a planet. Planet and ring geometry are schematic.

---

# Part 4

In [ ]:
# ============================================================
# WHAT THE DATA RULE OUT
# Inspired by Zuluaga et al. (2026)
# "Probing Exoplanetary Rings with Asterodensity Profiling:
#  A PhotoRing Analysis of Kepler-51"
#
# One self-contained Jupyter cell
# Requires: numpy, matplotlib, ffmpeg
#
# STORY
#   0–4.5 s    Many possible ring geometries
#   4.5–9 s    Compare them with Kepler transit constraints
#   9–13.2 s   Extreme configurations are progressively rejected
#   13.2–14 s  Transition
#   14–20 s    Final exclusion result
#
# IMPORTANT
# Ring geometries are schematic and do not reproduce individual
# posterior samples from the paper.
#
# The quoted credibility levels are results reported by the paper.
# "Excluded" here applies to the investigated class of extended,
# optically thick, high-obliquity ring configurations.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle, Ellipse
from IPython.display import Video, display


# ============================================================
# OUTPUT
# ============================================================

OUTPUT = "webm"       # "webm" or "mp4"

FPS = 30
DURATION = 20
NFRAMES = FPS * DURATION
DPI = 120

if OUTPUT == "webm":
    OUTFILE = "25234_what_the_data_rule_out.webm"
else:
    OUTFILE = "25234_what_the_data_rule_out.mp4"


# ============================================================
# HELPERS
# ============================================================

def clamp(x, a=0.0, b=1.0):
    return np.minimum(np.maximum(x, a), b)


def smoothstep(x):
    x = clamp(x)
    return x * x * (3.0 - 2.0 * x)


BG = "#02050b"


# ============================================================
# FIGURE
# ============================================================

fig = plt.figure(
    figsize=(12.8, 7.2),
    facecolor=BG
)


# ============================================================
# GLOBAL TITLE
# ============================================================

main_title = fig.text(
    0.5, 0.955,
    "WHAT THE DATA RULE OUT",
    ha="center",
    va="center",
    color="white",
    fontsize=21,
    fontweight="bold",
    zorder=200
)

main_subtitle = fig.text(
    0.5, 0.916,
    "Kepler photometry can reject some ring geometries without detecting rings",
    ha="center",
    va="center",
    color="#B7D6E8",
    fontsize=11,
    zorder=200
)


# ============================================================
# MAIN GEOMETRY PANEL
# ============================================================

ax = fig.add_axes(
    [0.04, 0.23, 0.92, 0.55],
    zorder=10
)

ax.set_facecolor(BG)

ax.set_xlim(
    0,
    4
)

ax.set_ylim(
    -0.90,
    0.90
)

ax.set_aspect("equal")
ax.axis("off")


# ============================================================
# FOUR SCHEMATIC RING CONFIGURATIONS
#
# 1 compact / moderately inclined
# 2 somewhat larger
# 3 extended / high-obliquity
# 4 extreme extended / high-obliquity
# ============================================================

centers = [
    0.50,
    1.50,
    2.50,
    3.50
]

planet_r = 0.14


# ------------------------------------------------------------
# CONFIGURATION 1
# compact
# ------------------------------------------------------------

p1 = Circle(
    (centers[0], 0.05),
    planet_r,
    facecolor="#3C7895",
    edgecolor="#DDF8FF",
    linewidth=1.2,
    zorder=12
)

r1_outer = Ellipse(
    (centers[0], 0.05),
    0.62,
    0.23,
    angle=-15,
    facecolor="#79C5DF",
    edgecolor="#D5F4FF",
    linewidth=1.2,
    alpha=0.31,
    zorder=9
)

r1_inner = Ellipse(
    (centers[0], 0.05),
    0.38,
    0.13,
    angle=-15,
    facecolor=BG,
    edgecolor="#739BAD",
    linewidth=0.8,
    zorder=10
)


# ------------------------------------------------------------
# CONFIGURATION 2
# larger but still relatively compact
# ------------------------------------------------------------

p2 = Circle(
    (centers[1], 0.05),
    planet_r,
    facecolor="#3C7895",
    edgecolor="#DDF8FF",
    linewidth=1.2,
    zorder=12
)

r2_outer = Ellipse(
    (centers[1], 0.05),
    0.80,
    0.31,
    angle=-27,
    facecolor="#79C5DF",
    edgecolor="#D5F4FF",
    linewidth=1.2,
    alpha=0.31,
    zorder=9
)

r2_inner = Ellipse(
    (centers[1], 0.05),
    0.43,
    0.16,
    angle=-27,
    facecolor=BG,
    edgecolor="#739BAD",
    linewidth=0.8,
    zorder=10
)


# ------------------------------------------------------------
# CONFIGURATION 3
# extended / high-obliquity
# ------------------------------------------------------------

p3 = Circle(
    (centers[2], 0.05),
    planet_r,
    facecolor="#3C7895",
    edgecolor="#DDF8FF",
    linewidth=1.2,
    zorder=12
)

r3_outer = Ellipse(
    (centers[2], 0.05),
    1.05,
    0.48,
    angle=-48,
    facecolor="#79C5DF",
    edgecolor="#D5F4FF",
    linewidth=1.3,
    alpha=0.34,
    zorder=9
)

r3_inner = Ellipse(
    (centers[2], 0.05),
    0.48,
    0.20,
    angle=-48,
    facecolor=BG,
    edgecolor="#739BAD",
    linewidth=0.8,
    zorder=10
)


# ------------------------------------------------------------
# CONFIGURATION 4
# extreme extended / high-obliquity
# ------------------------------------------------------------

p4 = Circle(
    (centers[3], 0.05),
    planet_r,
    facecolor="#3C7895",
    edgecolor="#DDF8FF",
    linewidth=1.2,
    zorder=12
)

r4_outer = Ellipse(
    (centers[3], 0.05),
    1.24,
    0.61,
    angle=-64,
    facecolor="#79C5DF",
    edgecolor="#D5F4FF",
    linewidth=1.3,
    alpha=0.37,
    zorder=9
)

r4_inner = Ellipse(
    (centers[3], 0.05),
    0.50,
    0.22,
    angle=-64,
    facecolor=BG,
    edgecolor="#739BAD",
    linewidth=0.8,
    zorder=10
)


all_patches = [
    r1_outer, r1_inner, p1,
    r2_outer, r2_inner, p2,
    r3_outer, r3_inner, p3,
    r4_outer, r4_inner, p4
]

for patch in all_patches:
    ax.add_patch(patch)


# ============================================================
# CONFIGURATION LABELS
# ============================================================

lab1 = ax.text(
    centers[0],
    -0.52,
    "COMPACT",
    ha="center",
    va="center",
    color="#C8F2FF",
    fontsize=11,
    fontweight="bold"
)

lab2 = ax.text(
    centers[1],
    -0.52,
    "MODERATE",
    ha="center",
    va="center",
    color="#C8F2FF",
    fontsize=11,
    fontweight="bold"
)

lab3 = ax.text(
    centers[2],
    -0.52,
    "EXTENDED",
    ha="center",
    va="center",
    color="#C8F2FF",
    fontsize=11,
    fontweight="bold"
)

lab4 = ax.text(
    centers[3],
    -0.52,
    "EXTREME",
    ha="center",
    va="center",
    color="#C8F2FF",
    fontsize=11,
    fontweight="bold"
)


sub1 = ax.text(
    centers[0],
    -0.68,
    "moderate tilt",
    ha="center",
    va="center",
    color="#809EAD",
    fontsize=8
)

sub2 = ax.text(
    centers[1],
    -0.68,
    "larger ring",
    ha="center",
    va="center",
    color="#809EAD",
    fontsize=8
)

sub3 = ax.text(
    centers[2],
    -0.68,
    "high obliquity",
    ha="center",
    va="center",
    color="#809EAD",
    fontsize=8
)

sub4 = ax.text(
    centers[3],
    -0.68,
    "large + high obliquity",
    ha="center",
    va="center",
    color="#809EAD",
    fontsize=8
)


# ============================================================
# STATUS SYMBOLS
# ============================================================

check1 = ax.text(
    centers[0],
    0.66,
    "POSSIBLE",
    ha="center",
    va="center",
    color="#9CEAFF",
    fontsize=11,
    fontweight="bold",
    alpha=0
)

check2 = ax.text(
    centers[1],
    0.66,
    "?",
    ha="center",
    va="center",
    color="#B7D5E3",
    fontsize=18,
    fontweight="bold",
    alpha=0
)

cross3 = ax.text(
    centers[2],
    0.66,
    "×",
    ha="center",
    va="center",
    color="#E8F7FF",
    fontsize=35,
    fontweight="bold",
    alpha=0
)

cross4 = ax.text(
    centers[3],
    0.66,
    "×",
    ha="center",
    va="center",
    color="#E8F7FF",
    fontsize=35,
    fontweight="bold",
    alpha=0
)


# ============================================================
# OBSERVATIONAL CONSTRAINT PANEL
# ============================================================

constraint_ax = fig.add_axes(
    [0.22, 0.105, 0.56, 0.105],
    zorder=20
)

constraint_ax.set_facecolor(
    "#06101A"
)

constraint_ax.set_xlim(
    0,
    1
)

constraint_ax.set_ylim(
    0,
    1
)

constraint_ax.axis("off")


constraint_title = constraint_ax.text(
    0.5,
    0.72,
    "ARCHIVAL KEPLER PHOTOMETRY",
    ha="center",
    va="center",
    color="#DDF5FF",
    fontsize=11,
    fontweight="bold",
    alpha=0
)

constraint_line = constraint_ax.text(
    0.5,
    0.30,
    "TRANSIT DEPTH • DURATION • GEOMETRY • ASTERODENSITY",
    ha="center",
    va="center",
    color="#8FC9DF",
    fontsize=9,
    fontweight="bold",
    alpha=0
)


# ============================================================
# PHASE TEXT
# ============================================================

phase_text = fig.text(
    0.5, 0.835,
    "",
    ha="center",
    va="center",
    color="white",
    fontsize=16,
    fontweight="bold",
    zorder=50
)

phase_sub = fig.text(
    0.5, 0.800,
    "",
    ha="center",
    va="center",
    color="#AFCBDC",
    fontsize=10,
    zorder=50
)


# ============================================================
# FINAL BACKGROUND
# ============================================================

result_bg = fig.add_axes(
    [0.025, 0.075, 0.95, 0.80],
    zorder=70
)

result_bg.set_facecolor(
    "#000207"
)

result_bg.set_xticks([])
result_bg.set_yticks([])

for spine in result_bg.spines.values():
    spine.set_visible(False)

result_bg.set_visible(False)
result_bg.set_alpha(0)


# ============================================================
# FINAL — SURVIVING COMPACT RING SYSTEM
# ============================================================

final_ring_ax = fig.add_axes(
    [0.065, 0.30, 0.27, 0.39],
    zorder=80
)

final_ring_ax.set_facecolor(
    "#000207"
)

final_ring_ax.set_xlim(
    -1,
    1
)

final_ring_ax.set_ylim(
    -1,
    1
)

final_ring_ax.set_aspect("equal")
final_ring_ax.axis("off")
final_ring_ax.set_visible(False)


fr_outer = Ellipse(
    (0, 0),
    1.55,
    0.55,
    angle=-17,
    facecolor="#7CC8E2",
    edgecolor="#D7F5FF",
    linewidth=1.6,
    alpha=0.30,
    zorder=4
)

fr_inner = Ellipse(
    (0, 0),
    0.91,
    0.31,
    angle=-17,
    facecolor="#000207",
    edgecolor="#779EAF",
    linewidth=0.9,
    zorder=5
)

fr_planet = Circle(
    (0, 0),
    0.32,
    facecolor="#3C7692",
    edgecolor="#DDF8FF",
    linewidth=1.5,
    zorder=7
)

fr_highlight = Circle(
    (-0.075, 0.085),
    0.16,
    facecolor="#72B5D1",
    edgecolor="none",
    alpha=0.32,
    zorder=8
)

final_ring_ax.add_patch(
    fr_outer
)

final_ring_ax.add_patch(
    fr_inner
)

final_ring_ax.add_patch(
    fr_planet
)

final_ring_ax.add_patch(
    fr_highlight
)


# ============================================================
# FINAL — RESULT PANELS
# ============================================================

b_ax = fig.add_axes(
    [0.39, 0.43, 0.52, 0.20],
    zorder=80
)

d_ax = fig.add_axes(
    [0.39, 0.20, 0.52, 0.20],
    zorder=80
)


for qax in [b_ax, d_ax]:

    qax.set_facecolor(
        "#030810"
    )

    qax.set_xlim(
        0,
        100
    )

    qax.set_ylim(
        0,
        1
    )

    qax.set_xticks(
        [0, 25, 50, 75, 100]
    )

    qax.set_yticks([])

    qax.tick_params(
        axis="x",
        colors="#8EA9B7",
        labelsize=8
    )

    for spine in qax.spines.values():
        spine.set_color("#355061")


# ============================================================
# KEPLER-51b
# ============================================================

b_ax.set_title(
    "KEPLER-51b",
    color="white",
    fontsize=13,
    fontweight="bold"
)

b_bar = b_ax.barh(
    [0.52],
    [99],
    height=0.22,
    color="#73D9F6",
    alpha=0.70
)[0]

b_ax.axvline(
    99,
    color="#D8F7FF",
    lw=1.3,
    alpha=0.85
)

b_value = b_ax.text(
    97,
    0.72,
    "99%",
    ha="right",
    va="center",
    color="#C8F5FF",
    fontsize=14,
    fontweight="bold"
)

b_caption = b_ax.text(
    50,
    0.19,
    "EXTREME RING CLASS EXCLUDED",
    ha="center",
    va="center",
    color="#91B9CA",
    fontsize=8,
    fontweight="bold"
)


# ============================================================
# KEPLER-51d
# ============================================================

d_ax.set_title(
    "KEPLER-51d",
    color="white",
    fontsize=13,
    fontweight="bold"
)

d_bar = d_ax.barh(
    [0.52],
    [70],
    height=0.22,
    color="#7BA7BA",
    alpha=0.65
)[0]

d_ax.axvline(
    70,
    color="#C6DCE6",
    lw=1.3,
    alpha=0.75
)

d_value = d_ax.text(
    68,
    0.72,
    "70%",
    ha="right",
    va="center",
    color="#D4E7EF",
    fontsize=14,
    fontweight="bold"
)

d_caption = d_ax.text(
    50,
    0.19,
    "EXTREME RING CLASS EXCLUDED",
    ha="center",
    va="center",
    color="#91A9B5",
    fontsize=8,
    fontweight="bold"
)


b_ax.set_visible(False)
d_ax.set_visible(False)


# ============================================================
# FINAL TEXT
# ============================================================

final_header = fig.text(
    0.5, 0.815,
    "SOME RINGS ARE ALREADY RULED OUT",
    ha="center",
    va="center",
    color="white",
    fontsize=18,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_sub = fig.text(
    0.5, 0.770,
    "Extended • optically thick • high-obliquity configurations",
    ha="center",
    va="center",
    color="#A9C9D8",
    fontsize=11,
    alpha=0,
    zorder=100
)


final_ring_title = fig.text(
    0.20, 0.265,
    "COMPACT RINGS",
    ha="center",
    va="center",
    color="#BDF3FF",
    fontsize=13,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_ring_note = fig.text(
    0.20, 0.220,
    "remain possible",
    ha="center",
    va="center",
    color="#91B3C2",
    fontsize=10,
    alpha=0,
    zorder=100
)


final_warning = fig.text(
    0.5, 0.115,
    "THE DATA CONSTRAIN RING GEOMETRY — THEY DO NOT DETECT RINGS",
    ha="center",
    va="center",
    color="#A7D7E9",
    fontsize=10,
    fontweight="bold",
    alpha=0,
    zorder=100
)


# ============================================================
# CLOCK / PROGRESS DIAL
# Decorative only — not a physical timescale
# ============================================================

clock_ax = fig.add_axes(
    [0.030, 0.035, 0.080, 0.14],
    facecolor="none",
    zorder=300
)

clock_ax.set_xlim(
    -1,
    1
)

clock_ax.set_ylim(
    -1,
    1
)

clock_ax.set_aspect("equal")
clock_ax.axis("off")

dial_radius = 0.82

dial_outer = Circle(
    (0, 0),
    dial_radius,
    facecolor="#07111A",
    edgecolor="#91DDF7",
    linewidth=1.3,
    alpha=0.80,
    zorder=30
)

dial_inner = Circle(
    (0, 0),
    dial_radius * 0.82,
    facecolor="none",
    edgecolor="#52788C",
    linewidth=0.7,
    alpha=0.65,
    zorder=31
)

clock_ax.add_patch(
    dial_outer
)

clock_ax.add_patch(
    dial_inner
)


for i in range(12):

    a = (
        2 *
        np.pi *
        i / 12
    )

    r1 = (
        dial_radius *
        (0.67 if i % 3 else 0.59)
    )

    r2 = (
        dial_radius *
        0.83
    )

    x1 = r1 * np.sin(a)
    y1 = r1 * np.cos(a)

    x2 = r2 * np.sin(a)
    y2 = r2 * np.cos(a)

    clock_ax.plot(
        [x1, x2],
        [y1, y2],
        color="#A7D8EA",
        lw=1.1 if i % 3 else 1.6,
        alpha=0.72,
        zorder=32
    )


hand_length = (
    dial_radius *
    0.56
)

dial_hand, = clock_ax.plot(
    [0, 0],
    [0, hand_length],
    color="#E8FAFF",
    lw=1.8,
    solid_capstyle="round",
    zorder=34
)

dial_pin = Circle(
    (0, 0),
    dial_radius * 0.065,
    facecolor="#E8FAFF",
    edgecolor="none",
    zorder=35
)

clock_ax.add_patch(
    dial_pin
)


# ============================================================
# UPDATE
# ============================================================

def update(frame):

    sec = frame / FPS
    t = frame / (NFRAMES - 1)


    # ========================================================
    # PHASE TEXT
    # ========================================================

    if sec < 4.5:

        phase_text.set_text(
            "MANY RING GEOMETRIES ARE POSSIBLE"
        )

        phase_sub.set_text(
            "Ring size, inclination and orientation can change the transit"
        )

    elif sec < 9.0:

        phase_text.set_text(
            "COMPARE WITH THE OBSERVED TRANSIT"
        )

        phase_sub.set_text(
            "Kepler photometry restricts the allowed parameter space"
        )

    else:

        phase_text.set_text(
            "EXTREME RINGS BECOME INCONSISTENT"
        )

        phase_sub.set_text(
            "Large, opaque and highly tilted configurations produce too much anomaly"
        )


    # ========================================================
    # SUBTLE PLANET / RING APPEARANCE
    # ========================================================

    intro = smoothstep(
        sec / 2.0
    )

    for patch in [
        p1, p2, p3, p4
    ]:
        patch.set_alpha(
            intro
        )

    for ring in [
        r1_outer,
        r2_outer,
        r3_outer,
        r4_outer
    ]:
        base_alpha = {
            id(r1_outer): 0.31,
            id(r2_outer): 0.31,
            id(r3_outer): 0.34,
            id(r4_outer): 0.37
        }[id(ring)]

        ring.set_alpha(
            base_alpha * intro
        )


    # ========================================================
    # OBSERVATIONAL CONSTRAINTS APPEAR
    # ========================================================

    constraint_p = smoothstep(
        (sec - 4.2) / 1.2
    )

    constraint_title.set_alpha(
        constraint_p
    )

    constraint_line.set_alpha(
        constraint_p
    )


    # ========================================================
    # PROGRESSIVELY REJECT EXTREME CONFIGURATIONS
    # ========================================================

    reject4 = smoothstep(
        (sec - 7.2) / 1.0
    )

    reject3 = smoothstep(
        (sec - 8.5) / 1.0
    )

    possible1 = smoothstep(
        (sec - 10.0) / 0.9
    )

    uncertain2 = smoothstep(
        (sec - 9.4) / 0.9
    )


    # extreme rings fade but remain visible
    r4_outer.set_alpha(
        0.37 *
        intro *
        (
            1.0 -
            0.72 * reject4
        )
    )

    r4_inner.set_alpha(
        intro *
        (
            1.0 -
            0.72 * reject4
        )
    )

    p4.set_alpha(
        intro *
        (
            1.0 -
            0.55 * reject4
        )
    )

    lab4.set_alpha(
        1.0 -
        0.55 * reject4
    )

    sub4.set_alpha(
        1.0 -
        0.55 * reject4
    )


    r3_outer.set_alpha(
        0.34 *
        intro *
        (
            1.0 -
            0.65 * reject3
        )
    )

    r3_inner.set_alpha(
        intro *
        (
            1.0 -
            0.65 * reject3
        )
    )

    p3.set_alpha(
        intro *
        (
            1.0 -
            0.50 * reject3
        )
    )

    lab3.set_alpha(
        1.0 -
        0.50 * reject3
    )

    sub3.set_alpha(
        1.0 -
        0.50 * reject3
    )


    cross4.set_alpha(
        reject4
    )

    cross3.set_alpha(
        reject3
    )

    check1.set_alpha(
        possible1
    )

    check2.set_alpha(
        uncertain2
    )


    # ========================================================
    # FINAL TRANSITION
    # ========================================================

    scene2 = smoothstep(
        (sec - 13.2) / 0.8
    )

    scene1_alpha = (
        1.0 -
        scene2
    )


    ax.set_alpha(
        scene1_alpha
    )

    constraint_ax.set_alpha(
        scene1_alpha
    )

    phase_text.set_alpha(
        scene1_alpha
    )

    phase_sub.set_alpha(
        scene1_alpha
    )

    main_title.set_alpha(
        scene1_alpha
    )

    main_subtitle.set_alpha(
        scene1_alpha
    )


    # ========================================================
    # FINAL BACKGROUND
    # ========================================================

    if scene2 > 0.005:

        result_bg.set_visible(
            True
        )

        result_bg.set_alpha(
            0.98 * scene2
        )

        final_ring_ax.set_visible(
            True
        )

        final_ring_ax.set_alpha(
            scene2
        )

        b_ax.set_visible(
            True
        )

        b_ax.set_alpha(
            scene2
        )

        d_ax.set_visible(
            True
        )

        d_ax.set_alpha(
            scene2
        )

    else:

        result_bg.set_visible(
            False
        )

        final_ring_ax.set_visible(
            False
        )

        b_ax.set_visible(
            False
        )

        d_ax.set_visible(
            False
        )


    # ========================================================
    # FINAL TEXT
    # ========================================================

    final_header.set_alpha(
        scene2
    )

    final_sub.set_alpha(
        scene2
    )

    final_ring_title.set_alpha(
        scene2
    )

    final_ring_note.set_alpha(
        scene2
    )

    final_warning.set_alpha(
        scene2
    )


    # ========================================================
    # CLOCK
    # ========================================================

    dial_angle = (
        2 *
        np.pi *
        t
    )

    hand_x = (
        hand_length *
        np.sin(
            dial_angle
        )
    )

    hand_y = (
        hand_length *
        np.cos(
            dial_angle
        )
    )

    dial_hand.set_data(
        [0, hand_x],
        [0, hand_y]
    )


    return []


# ============================================================
# ANIMATION
# ============================================================

anim = FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000 / FPS,
    blit=False
)


# ============================================================
# EXPORT
# ============================================================

if OUTPUT == "webm":

    writer = FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2400,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-row-mt", "1",
            "-deadline", "good",
            "-cpu-used", "2"
        ]
    )

else:

    writer = FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=4000,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-preset", "medium",
            "-movflags", "+faststart"
        ]
    )


anim.save(
    OUTFILE,
    writer=writer,
    dpi=DPI
)

plt.close(fig)

print(
    f"Saved: {OUTFILE}"
)

display(
    Video(
        OUTFILE,
        embed=True,
        html_attributes="controls loop"
    )
)

# Какие кольца у Kepler-51 уже можно исключить?

Поиск колец у экзопланет не обязательно должен закончиться их обнаружением. Даже если кольца невозможно непосредственно увидеть, наблюдения позволяют определить, какие их размеры и ориентации совместимы с транзитом, а какие должны были бы оставить слишком сильный след.

В случае Kepler-51 авторы проверили широкий набор возможных геометрий колец. Особенно заметный PhotoRing-эффект должны создавать протяжённые, оптически толстые системы колец с высокой обликвитетностью. Такие кольца существенно изменяют наблюдаемую геометрию транзита и приводят к слишком большому расхождению между плотностью звезды, полученной из транзита, и её независимо определённой плотностью.

Архивная фотометрия Kepler уже позволяет исключить исследованный класс таких экстремальных конфигураций с достоверностью около 99% для Kepler-51b и 70% для Kepler-51d. При этом более компактные и умеренно наклонённые системы колец остаются совместимыми с данными. 2609.25234v1.pdf

Это важный отрицательный результат: наблюдения пока не показывают, что кольца у Kepler-51 существуют, но уже начинают определять, какими эти кольца не могут быть.

Дополнительное ограничение даёт JWST. Наблюдения Kepler-51d не обнаружили явного фотометрического сигнала крупного непрозрачного кольца, а спектр пропускания этой планеты лучше объясняется высоко расположенной атмосферной дымкой. Авторы поэтому считают массивное непрозрачное кольцо у Kepler-51d маловероятным, хотя более прозрачные или пористые кольца полностью не исключены. 2609.25234v1.pdf

Изображённые в анимации системы колец схематичны и не являются отдельными моделями из posterior distribution авторов. Значения 99% и 70% относятся к ограничениям на исследованный класс протяжённых, оптически толстых и сильно наклонённых конфигураций.

# Which Rings Can Already Be Ruled Out Around Kepler-51?

A search for exoplanetary rings does not have to end with a detection. Even when rings cannot be directly resolved, observations can determine which ring sizes and orientations remain compatible with a transit and which should have produced a much stronger signature.

For Kepler-51, the authors explored a broad range of possible ring geometries. Particularly strong PhotoRing effects are expected from extended, optically thick ring systems viewed at high obliquity. Such configurations substantially alter the observed transit geometry and produce an excessive discrepancy between the stellar density inferred from the transit and the independently determined stellar density.

Archival Kepler photometry is already sufficient to exclude the investigated class of these extreme configurations at approximately 99% credibility for Kepler-51b and 70% for Kepler-51d. More compact and moderately inclined ring systems, however, remain compatible with the data. 2609.25234v1.pdf

This is an important negative result: the observations do not demonstrate that Kepler-51 possesses rings, but they are beginning to tell us what those rings cannot look like.

JWST provides an additional constraint. Observations of Kepler-51d show no clear photometric signature of a large opaque ring, while its transmission spectrum is better explained by high-altitude atmospheric haze. The authors therefore regard a massive opaque exoring around Kepler-51d as unlikely, although more transparent or porous rings are not completely excluded. 2609.25234v1.pdf

The ring systems shown in the animation are schematic and do not represent individual posterior solutions from the study. The 99% and 70% values refer to constraints on the investigated class of extended, optically thick, high-obliquity ring configurations.